# Build Measured Intrinsic Wavefront (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-04-28
**Last Modified:** 2026-05-12
**Status:** Draft
**Keywords:** AOS, intrinsic, double-Zernike, FAM, focal plane, OFC, U-mode, reachability

## Description

Build an empirical estimate of the focal-plane intrinsic Zernike map
("measured intrinsic") from FAM donut data, by fitting and subtracting
a chosen subset of Double-Zernike modes (focal `k` × pupil `j`).

Two complementary subtraction paths are run side-by-side, both
sharing the same per-visit DZ fit machinery and iterative refinement:

### Path A — U-mode constrained

1. Per visit, fit the *full* DZ grid (k = `k_min`..`k_max`, all 21
   pupil `j` in `iZs`) to `data − tabulated_intrinsic`.
2. Stack the resulting `n_k × n_j` coefficient vector `w` for the
   visit and project it onto the first `n_keep` left-singular
   vectors of the OFC sensitivity matrix `S` (sliced to the same
   k / j range):
   $$ w_\mathrm{fit} \;=\; U_\mathrm{eff}\, U_\mathrm{eff}^\top\, w,
      \qquad U_\mathrm{eff} = U[:,\,:n_\mathrm{keep}]. $$
3. Rebuild the per-donut DZ contribution from the *projected*
   coefficients and subtract it from the per-donut wavefront, then
   median-bin onto a focal-plane grid for iter-1's measured intrinsic.
4. Iterate `n_iter` times, feeding the previous iteration's grid back
   in as the tabulated intrinsic.

Path A's subtraction is intrinsically constrained to the actuator
subspace OFC can correct, so DoF-degenerate modes remain in the
residual rather than being absorbed into the per-visit fit.

### Path B — Reachability-thresholded

1. Build a sparse `(k, j)` removal spec by selecting every cell with
   reachability $f_{k,j} = \|U_\mathrm{eff}^\top e_{k,j}\|^2 \ge$
   `reach_threshold`.  (Equivalent to picking a hard cutoff on the
   per-DZ leverage that the kept U-modes have.)
2. Per visit, fit *only* those `(k, j)` cells to
   `data − tabulated_intrinsic`, subtract the contribution, and
   median-bin per pupil j.
3. Iterate `n_iter` times.

Path B mirrors the original removal-spec approach with the spec
chosen automatically from the OFC SVD (overridable via
`removal_spec_manual`).

Both paths run on the same multi-chunk donut dataset and produce
their own DZ-fit tables, iterated measured-intrinsic grids, and a
per-pupil-j iteration-progression page set; a comparison section
overlays the iter-final A and B maps and their A−B difference.

Key functionality:
1. Load donut + visit-info parquet pairs across all chunks for the
   chosen `binning`, filter by day_obs / elevation / rotator angle /
   seq_num.
2. SVD of the OFC sensitivity matrix `S` for diagnostic plots and to
   feed Path A / Path B.
3. Run Path A (U-mode constrained) and Path B (reachability spec) in
   parallel, each iterating `n_iter` times.
4. Per-pupil-j iter-progression plots showing Original, every iter,
   and the tabulated intrinsic on a shared 5–95 % colour scale.
5. Path A vs Path B difference maps with per-j RMS summary.
6. Per-DZ-term tracking (iter-1 vs iter-final) and per-visit
   validation diagnostics for Path B.
7. Save per-path DZ fits and measured-intrinsic grids.

**Output:** under
`output/build_measured_intrinsic/{binning}_nkeep_{n_keep}_…/`, files
prefixed `measured_intrinsic_{output_label}_pathA_*` /
`measured_intrinsic_{output_label}_pathB_*`.

**Based on:** `code/measured_intrinsic.py`, `code/dz_fitting.py`,
`code/intrinsics_lib.py`, the `run_pipeline` mktable / fit / plots
flow, and `jk_coverage_plots.ipynb` for the SVD / Reachability
reference analysis.


## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-28 | Aaron Roodman | Initial version. |
| 2026-05-12 | Aaron Roodman | Major rework: (1) multi-chunk loading driven by `binning` ("bin_2x" default) resolved against `runs.yaml`, with ±3° rotator filter default and any elevation; (2) new OFC SVD / Reachability reference section — V, σ, U, U² matrix plots, reachability heatmap with cell text, per-j summary table, and a reference fit demo on a real visit; (3) **Path A** — `build_measured_intrinsic_uconstrained` fits all (k=1..6) × 21 pupil-j DZ coefficients per visit and projects each visit's coefficient vector onto the first `n_keep` U-modes (`w_proj = U_eff @ U_effᵀ @ w`), iterating `n_iter` times; (4) **Path B** — `build_measured_intrinsic` with `removal_spec` built automatically from `removal_spec_from_reachability` at `reach_threshold` (overridable via `removal_spec_manual`); (5) **Path A per-visit validation** — heatmaps of `W_raw`, `W_fit`, `W_residual` vs visit ordinal; U-mode amplitude heatmap and line plots; per-pupil-j panel plots of `W_fit` and `W_residual` (lines coloured by focal k); (6) **Path A vs Path B comparison** — side-by-side iter-final measured-intrinsic maps and per-pupil-j A−B differences with RMS summary; (7) outputs reorganised under `output/build_measured_intrinsic/{binning}_nkeep_{n_keep}_…/` with every file prefixed `measured_intrinsic_{output_label}_…`. |
| 2026-05-12 (v2) | Aaron Roodman | Header now describes both paths; default `n_iter = 3` and `reach_threshold = 0.09`; Path B printout shows the explicit pupil-j list per k plus cells within ±0.05 of threshold; per-pupil-j iter-progression plots are now adaptive (Original | Iter 1 | … | Iter n | Tabulated) and a parallel Path A plots cell was added; tracking-cell now compares iter-1 vs iter-final; downstream `iter2_grids` always refers to the final iteration. |
| 2026-05-14 (v3) | Aaron Roodman | Major restructure: visit filters now include elevation default 55–75° and allowed_bands default [r, i, z]; Z4 height correction adjusts ONLY the intrinsic via `intrinsic_new = intrinsic_orig + Z4hgt − Z4hgt_intrinsic`, with a new validation map section; OFC SVD plots number modes from 1 and the V plot now annotates the 50 DoF with M2 Hex / Cam Hex / M1M3 / M2 groups; sections re-ordered to **Path A impl → Path A validation → Path B impl → Path B validation → A vs B comparison**; each path now produces its own validation block (w heatmaps, 22-panel per-(k,j) vs visit with `(1,4)` Focus split, iter (n−1) vs n stability heatmap, example-visit histograms, σ / σ_MAD vs visit) plus a comparison PDF; Path A also gets the U-mode amplitude diagnostics for all `n_keep` modes; old standalone tracking / robust-RMS / per-visit-validation cells removed (folded into the per-path validation cells); tqdm switched from `tqdm.auto` to plain `tqdm` to avoid the "Error displaying widget: model not found" message. |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access — multi-chunk + visit filters](#data)
5. [Z4 Height Correction (precompute, intrinsic-only)](#z4height)
6. [Z4 Height Correction — Validation](#z4val)
7. [OFC Sensitivity SVD & Reachability (reference)](#svd)
    - [7.1 SVD diagnostic plots — V, σ, U, U²](#svd-plots)
    - [7.2 Reachability heatmap & summary table](#reachability)
    - [7.3 Reference fit demo](#fit-demo)
8. [Path A — U-mode constrained DZ removal](#analysis-pathA)
9. [Path A — Validation](#pathA-validation)
10. [Path B — Reachability-thresholded DZ removal](#analysis-pathB)
11. [Path B — Validation](#pathB-validation)
12. [Path A vs Path B — Measured Intrinsic comparison](#compare-AB)
13. [Iter-final (Path B) Measured Intrinsic — CCS-binned maps](#ccs-maps)
14. [Iter-final (Path B) Measured Intrinsic — OCS, rotator near 0](#ocs-rot0-maps)
15. [Output Tables](#output)
16. [Output Format Options](#format)


<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Binning selector -----
#   binning = 'bin_2x'  ->  use the three bin_2x chunks (chunk1, 3, 5)
#   binning = 'bin_1x'  ->  use the three bin_1x chunks (chunk2, 4, 6)
# Explicit `donut_parquets` (below) takes precedence if provided.
binning = 'bin_2x'

# ----- Input donut parquet files (one per pipeline chunk) -----
# If None, the data cell resolves the three chunks for `binning`
# from runs.yaml.  Each donut parquet must have its matching
# `*_visits.parquet` sidecar next to it.
donut_parquets = None         # list[str] or None

# Coordinate system for the fit and grid (FAM triplets are taken near
# rotator angle = 0, so OCS and CCS are essentially equivalent).
coord_sys = 'OCS'

# ----- Filters (None = no constraint) -----
day_obs_min   = None
day_obs_max   = None
alt_min_deg   = 55.0        # elevation in degrees
alt_max_deg   = 75.0
rot_min_deg   = -3.0        # rotator_angle in degrees
rot_max_deg   = +3.0
seq_num_min   = None
seq_num_max   = None
# Photometric bands to keep (None = no constraint).
allowed_bands = ['r', 'i', 'z']

# ----- Per-visit quality cuts (matches run_pipeline fit step) -----
min_donuts        = 500     # require at least this many donuts per visit
bad_fit_threshold = 2.0     # flag bad if any |coeff| > this (μm)

# Cap on the number of visits actually analysed.  Useful for fast
# end-to-end iteration through the notebook.  0 = no cap (use every
# visit that survives the upstream filters).  When > 0 we keep an
# evenly-spaced subset of visits across the full filtered range.
max_visits = 0

# ============================================================
# OFC sensitivity-matrix / SVD parameters
# ============================================================
# The U-mode constrained path (Path A) and the reachability-thresholded
# path (Path B) both rest on the SVD of the OFC sensitivity matrix S
# (sliced to the k, j of interest) after right-multiplication by the
# OFC normalization matrix.  These parameters set up that SVD.

# Path to the OFC normalization-weights yaml (50-DoF baseline).
# The weights live in ts_config_mttcs (not ts_ofc) at
#   MTAOS/ofc/normalization_weights/range0.5_fwhm-0.15.yaml
# When this parameter is None, the SVD cell auto-discovers it from
# $TS_CONFIG_MTTCS_DIR (the EUPS env var set by
# 'setup ts_config_mttcs').
ofc_normalization_yaml = None

# Focal-Noll k slice (k = 1..6 by default).
k_min, k_max = 1, 6

# Number of U-modes (singular vectors) kept for the projection.
# Equivalent to truncating S to its top n_keep singular components.
n_keep = 25

# Reachability threshold (path B): a (k, j) cell is selected for
# removal if its reachability fraction f_{k,j} >= reach_threshold.
reach_threshold = 0.09

# ----- Removal spec (path B override) -----
# If None, the analysis cell builds the removal spec automatically
# from the reachability grid for the current `reach_threshold`.
# Provide an explicit dict here to override.
removal_spec_manual = None

# ----- Iteration -----
n_iter = 3

# ----- Focal-plane grid -----
n_bins = 73                  # 18*4 + 1 trio-style binning
fp_radius_basis = 1.75       # field radius for Z basis normalization
fp_radius_grid  = 1.8        # grid extent for binned medians

# ----- Z4 CCD-height correction -----
# When `height_map_fits` is set, Z4 is corrected per donut BEFORE the
# DZ fits and the binning:
#   - measured Z4: subtract Z4hgt        = 15 μm/mm × height(fpx, fpy)
#   - tabulated Z4: subtract Z4hgt_transpose
# `Z4hgt_transpose` evaluates the height at the per-CCD (x<->y)
# transpose of (fpx, fpy), to match the bug in the pipeline's
# per-CCD intrinsic Zernike calculation.  Set
# intrinsic_transpose_bug=False once the upstream tabulation is fixed.
height_map_fits          = 'output/LSST_FP_cold_b_measurement_4col_bysurface.fits'
height_to_z4_factor      = None   # None -> ccd_height.HEIGHT_TO_Z4_UM_PER_MM
intrinsic_transpose_bug  = True   # use per-CCD transposed Z4hgt for tabulated

# ----- Per-visit residual validation -----
# Histograms are cheap; the residual color maps are slow because
# `binned_statistic_2d` is called per visit per j.  Default to
# histograms only.
make_residual_maps   = False
residual_j_range     = range(4, 20)     # pupil j = 4..19 (16 panels = 4x4 fully filled)
residual_hist_range  = (-1.0, 1.0)      # μm
residual_n_hist_bins = 60
residual_map_n_bins  = 16               # coarse, like the movie binning
residual_map_fp_radius = 1.8

# Iter-2 measured-intrinsic CCS map pages.  These rebin the
# OCS-frame DZ-subtracted zk values on a (thx_CCS, thy_CCS)
# grid so CCD-fixed features stand out.
ccs_maps_per_page  = 4    # 2x2 layout per page
ccs_map_n_bins     = None # None -> reuse n_bins from §6 (default 73)
ccs_map_fp_radius  = None

# Half-width (deg) for the iter-2 "OCS, rotator near zero"
# map view (§9).  Donuts whose visit has |rotator_angle|
# > rot0_window_deg are dropped before the OCS rebin so
# that view is rotator-smearing-free.
rot0_window_deg    = 3.0 # None -> reuse fp_radius_grid (default 1.8)

# Subsample the per-visit histogram (and map) pages: only
# emit a page every `hist_visit_stride` visits.  Summary
# σ / σ_MAD stats are still computed for every good
# visit and all of those drive the time-series plots.
hist_visit_stride = 12
per_zernike_sigma_j = range(4, 9)        # j = 4..8

# ----- Output -----
output_dir = None            # None -> derive from donut stem + filter tag
output_format = 'parquet'    # parquet (recommended)


<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable, vstack as table_vstack

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

sys.path.insert(0, 'code')
from dz_fitting import derive_noll_indices
from measured_intrinsic import (
    expand_removal_spec,
    apply_visit_filters,
    bin_median_focal,
    interpolate_grid_at_donuts,
    build_measured_intrinsic,
    build_measured_intrinsic_uconstrained,
    removal_spec_from_reachability,
    assemble_intrinsic_table,
    save_dz_fits,
    _fit_one_image_subset,
    _dz_contrib_from_params,
)

from intrinsics_lib import (
    classify_visit, visit_marker_style,
    build_visit_marker_lookup, markers_legend_figure,
)
from run_pipeline import (
    load_runs as _rp_load_runs,
    load_param_sets as _rp_load_param_sets,
    resolve_run as _rp_resolve_run,
    donut_path as _rp_donut_path,
)
# OFC sensitivity matrix + normalization weights (needed for the SVD
# section that drives the U-mode and reachability paths).  Both are
# only available inside the LSST conda env on RSP; on the laptop we
# fall back to None and skip those cells.
try:
    import yaml
    from lsst.ts.ofc import OFCData
    _ofc_ok = True
except Exception as _e:
    print(f'(OFC / yaml import skipped: {type(_e).__name__}: {_e})')
    _ofc_ok = False

# CCD focal-plane height map helpers (for the Z4 correction).  These
# imports pull in lsst.obs.lsst, sklearn, and astropy.io — they only
# work on RSP / inside the LSST conda env.
try:
    from ccd_height import (
        compute_fp_coords,
        make_metrology_table,
        get_height_interpolator,
        height_to_z4,
        transpose_around_ccd_centers,
        HEIGHT_TO_Z4_UM_PER_MM,
    )
    _ccd_height_ok = True
except Exception as _e:
    print(f'(ccd_height import skipped: {type(_e).__name__}: {_e})')
    _ccd_height_ok = False

print('Loaded measured_intrinsic helpers.')


<a id='functions'></a>
## 3. Helper Functions

In [ ]:
def filter_donuts_by_visits(donut_df, visits_kept):
    """Return rows of donut_df whose (day_obs, seq_num) appears in visits_kept."""
    keep_keys = set(zip(
        np.asarray(visits_kept['day_obs']).tolist(),
        np.asarray(visits_kept['seq_num']).tolist(),
    ))
    keys = list(zip(donut_df['day_obs'].tolist(),
                    donut_df['seq_num'].tolist()))
    mask = np.array([k in keep_keys for k in keys])
    return donut_df.loc[mask].reset_index(drop=True)


def common_color_scale(grids, plo=5.0, phi=95.0):
    """Return (vmin, vmax) covering the plo-phi percentile of *all* grids
    (skip NaNs; ignore empty grids)."""
    pooled = np.concatenate(
        [g.ravel()[~np.isnan(g.ravel())] for g in grids if g is not None
         and np.any(np.isfinite(g))])
    if pooled.size == 0:
        return -1.0, 1.0
    return tuple(np.nanpercentile(pooled, [plo, phi]))


def output_dir_default(binning, n_keep_label, coord_sys,
                       day_obs_min, day_obs_max,
                       seq_num_min, seq_num_max,
                       alt_min_deg, alt_max_deg,
                       rot_min_deg, rot_max_deg):
    """Build a self-describing output subdirectory.

    Top-level: `output/build_measured_intrinsic/`
    Tag: `{binning}_nkeep_{n_keep_label}_{coord_sys}` plus filter
    qualifiers as needed.
    """
    parts = [str(binning), f'nkeep_{n_keep_label}', str(coord_sys)]
    if day_obs_min or day_obs_max:
        parts.append(f'day_{day_obs_min or ""}_{day_obs_max or ""}')
    if seq_num_min or seq_num_max:
        parts.append(f'seq_{seq_num_min or ""}_{seq_num_max or ""}')
    if alt_min_deg is not None or alt_max_deg is not None:
        parts.append(f'alt_{alt_min_deg or ""}_{alt_max_deg or ""}')
    if rot_min_deg is not None or rot_max_deg is not None:
        parts.append(f'rot_{rot_min_deg or ""}_{rot_max_deg or ""}')
    return Path('output') / 'build_measured_intrinsic' / '_'.join(parts)


# ----------------------------------------------------------------------
# Multi-chunk donut parquet resolution
# ----------------------------------------------------------------------

def resolve_chunk_parquets(binning, donut_parquets=None):
    """Return the list of donut parquet paths to use.

    If `donut_parquets` is provided, that wins.  Otherwise we look up
    the runs in `runs.yaml` whose `param_set` matches the requested
    binning (`fam_danish_v1_triplets_bin_2x` or `..._bin_1x`) and
    return the donut paths in chunk order.
    """
    if donut_parquets:
        return [Path(p) for p in donut_parquets]

    target_ps = f'fam_danish_v1_triplets_{binning}'
    runs = _rp_load_runs().get('runs', {})
    param_sets = _rp_load_param_sets()

    paths = []
    for name, cfg in runs.items():
        if cfg.get('param_set') != target_ps:
            continue
        resolved = _rp_resolve_run(cfg, param_sets)
        p = Path(_rp_donut_path(resolved))
        if p.exists():
            paths.append(p)
        else:
            print(f'  (skipping {name}: {p} not found)')
    if not paths:
        raise FileNotFoundError(
            f'No donut parquet files resolved for binning={binning!r}')
    return paths


def visits_sidecar_path(donut_parquet_path):
    """`{stem}_visits.parquet` next to the donut parquet."""
    p = Path(donut_parquet_path)
    return p.with_name(p.stem + '_visits.parquet')


def load_chunks(donut_parquet_paths):
    """Read donut + visits sidecars across chunks, return (donut_df, visits)."""
    donut_frames = []
    visits_tables = []
    for p in donut_parquet_paths:
        vp = visits_sidecar_path(p)
        assert p.exists(), f'missing {p}'
        assert vp.exists(), f'missing {vp}'
        d = pd.read_parquet(p)
        v = QTable.read(str(vp), format='parquet')
        donut_frames.append(d)
        visits_tables.append(v)
        print(f'  {p.name}:  {len(d):>8d} donuts,  {len(v):>4d} visits')
    donut_df = pd.concat(donut_frames, ignore_index=True)
    visits_full = (visits_tables[0]
                   if len(visits_tables) == 1
                   else table_vstack(visits_tables, join_type='exact'))
    return donut_df, visits_full


def plot_iter_progression_for_j(j, original_grid, iter_grids,
                                tabulated_grid, xbins, ybins,
                                coord_sys, plo=5.0, phi=95.0,
                                path_tag=''):
    """Per-pupil-j N-panel page: Original | Iter 1 | ... | Iter n | Tabulated.

    `iter_grids` is a list of ``(label, grid)`` tuples, one per iteration
    in order.  Panel 1 (Original) uses its own 5-95 % range; all other
    panels share a single pooled 5-95 % range across the iteration
    grids and the Tabulated grid so they are directly comparable.
    """
    panels = [('Original median (raw zk)', original_grid)]
    panels.extend(iter_grids)
    panels.append(('Tabulated intrinsic', tabulated_grid))

    if original_grid is not None and np.any(np.isfinite(original_grid)):
        vmin0, vmax0 = np.nanpercentile(original_grid, [plo, phi])
    else:
        vmin0, vmax0 = -1.0, 1.0
    shared = [g for _, g in iter_grids] + [tabulated_grid]
    vmin_i, vmax_i = common_color_scale(shared, plo=plo, phi=phi)

    n_panels = len(panels)
    ncols = min(n_panels, 3) if n_panels > 4 else 2
    nrows = (n_panels + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.5 * ncols, 4.2 * nrows),
                             layout='constrained')
    axes = np.asarray(axes).reshape(nrows, ncols).ravel()
    for idx, ax in enumerate(axes):
        if idx >= n_panels:
            ax.set_visible(False)
            continue
        title, grid = panels[idx]
        if grid is None or not np.any(np.isfinite(grid)):
            ax.set_visible(False)
            continue
        if idx == 0:
            vmin, vmax = vmin0, vmax0
            scale_note = f'5-95% own = [{vmin0:.3f}, {vmax0:.3f}] μm'
        else:
            vmin, vmax = vmin_i, vmax_i
            scale_note = f'shared 5-95% = [{vmin_i:.3f}, {vmax_i:.3f}] μm'
        im = ax.imshow(grid.T, origin='lower',
                       extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
                       cmap='viridis', interpolation='none',
                       aspect='equal',
                       vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8, label='μm')
        ax.set_xlabel(f'thy_{coord_sys} [deg]')
        ax.set_ylabel(f'thx_{coord_sys} [deg]')
        ax.set_title(f'{title}\n({scale_note})', fontsize=10)
    suptitle = (f'Pupil Z{j}  ({coord_sys}){path_tag}  '
                f'— Original uses own 5-95 %; iterations + Tabulated share '
                f'a single 5-95 %')
    fig.suptitle(suptitle, fontsize=12)
    return fig


def _build_iter_grids_list(result_obj):
    """Return ``[('Iter-1 measured', grid_j_dict), ...]`` in order."""
    return [(f'Iter-{i + 1} measured', it['measured_grid'])
            for i, it in enumerate(result_obj['iter_results'])]


def filter_visits_by_band(visits_table, allowed_bands):
    """Drop visits whose `band` column is not in `allowed_bands`.

    `allowed_bands` of None or [] means no band filter.  Visits with a
    missing/empty band string are kept only if `allowed_bands` is None.
    """
    if not allowed_bands or 'band' not in visits_table.colnames:
        return visits_table
    bands = np.asarray(visits_table['band']).astype(str)
    keep = np.array([b in set(allowed_bands) for b in bands])
    return visits_table[keep]


# ======================================================================
# Validation helpers (used by both Path A and Path B validation cells).
# ======================================================================

def stack_per_visit_coeffs(fit_rows, kj_list):
    """Stack a list of per-visit param dicts into a (n_visits, n_kj) array.

    `kj_list` is the row order (a list of ``(k, j)`` tuples).  Missing
    entries are NaN.  Bad-flagged visits are kept (they will appear as
    NaN columns if the fit didn't populate the keys).
    """
    n_v = len(fit_rows)
    n_m = len(kj_list)
    W = np.full((n_v, n_m), np.nan)
    for v, row in enumerate(fit_rows):
        for m, (k, j) in enumerate(kj_list):
            W[v, m] = float(row.get(f'dz_z{j}_c{k}', np.nan))
    return W


def removal_spec_kj_list(by_focal):
    """Flatten a `{k: [j, ...]}` removal spec into a sorted list of
    ``(k, j)`` tuples (k outer, j inner)."""
    return sorted(((int(k), int(j))
                   for k, js in by_focal.items() for j in js),
                  key=lambda kj: (kj[0], kj[1]))


def plot_w_heatmap(W, kj_list, n_k_for_lines, n_j_for_lines,
                   title, vmax=None, k_min=None):
    """Heatmap of (visit ordinal, kj-row) for a coefficient stack.

    `n_k_for_lines`, `n_j_for_lines`, and `k_min` are used to draw
    k-boundary lines when the kj_list happens to be the contiguous
    n_k × n_j grid; otherwise the dividers are skipped.
    """
    n_v, n_m = W.shape
    if vmax is None:
        finite = W[np.isfinite(W)]
        vmax = float(np.nanpercentile(np.abs(finite), 98)) if finite.size else 1e-3
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1e-3
    fig, ax = plt.subplots(figsize=(11.5, 7.0), layout='constrained')
    im = ax.imshow(W.T, aspect='auto', cmap='RdBu_r',
                   vmin=-vmax, vmax=+vmax,
                   extent=[-0.5, n_v - 0.5, n_m - 0.5, -0.5])
    ax.set_xlabel('Visit ordinal')
    ax.set_ylabel('(k, j) row')
    ax.set_title(title)
    # y tick labels per cell (small)
    ax.set_yticks(np.arange(n_m))
    ax.set_yticklabels([f'({k},{j})' for k, j in kj_list], fontsize=6)
    plt.colorbar(im, ax=ax, label='μm')
    return fig


def _per_j_panel_layout(iZs_arr):
    """Page layout: panel 0 = (1,4) alone, panel 1 = (k=2..6 of j=4),
    panels 2..21 = each remaining j in iZs_arr with k = 1..6."""
    js_rest = [int(j) for j in iZs_arr if int(j) != 4]
    return js_rest                              # 20 j values


def plot_per_kj_vs_visit_page(W, kj_list, iZs_arr,
                              k_min, k_max,
                              title_root='', iter_label=''):
    """Build a 22-panel page (DZ-fit vs visit ordinal) for ONE iteration.

    Layout:
      panel 0     — (k=1, j=4) Focus alone (1 line)
      panel 1     — j=4, k=2..6 overlaid
      panels 2..21 — each remaining j in `iZs_arr`, k=1..6 overlaid

    A k=1..6 legend is drawn beneath the grid.
    """
    js_rest = [int(j) for j in iZs_arr if int(j) != 4]
    ncols = 4
    nrows = 6
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 14),
                             layout='constrained', sharex=True)
    axes = axes.ravel()
    cmap_k = plt.get_cmap('viridis')

    def _kj_col(k_val, j_val):
        try:
            return kj_list.index((int(k_val), int(j_val)))
        except ValueError:
            return None

    n_v = W.shape[0]
    # Panel 0: (k=1, j=4) alone
    ax = axes[0]
    col = _kj_col(1, 4)
    if col is not None:
        ax.plot(np.arange(n_v), W[:, col], '.', ms=4, color=cmap_k(0.0))
    ax.set_title('(k=1, j=4) — Focus', fontsize=9)
    ax.axhline(0, color='k', lw=0.4, alpha=0.5); ax.grid(alpha=0.3)

    # Panel 1: (k=2..6, j=4)
    ax = axes[1]
    for k_val in range(2, k_max + 1):
        col = _kj_col(k_val, 4)
        if col is None:
            continue
        c = cmap_k((k_val - k_min) / max(1, k_max - k_min))
        ax.plot(np.arange(n_v), W[:, col], '.', ms=3, color=c)
    ax.set_title('j=4, k=2..6', fontsize=9)
    ax.axhline(0, color='k', lw=0.4, alpha=0.5); ax.grid(alpha=0.3)

    # Panels 2..21: each remaining j with k = 1..6
    for pidx, j_val in enumerate(js_rest, start=2):
        if pidx >= len(axes):
            break
        ax = axes[pidx]
        for ki, k_val in enumerate(range(k_min, k_max + 1)):
            col = _kj_col(k_val, j_val)
            if col is None:
                continue
            c = cmap_k(ki / max(1, k_max - k_min))
            ax.plot(np.arange(n_v), W[:, col], '.', ms=2.5, color=c)
        ax.set_title(f'j={int(j_val)}, k=1..6', fontsize=8)
        ax.axhline(0, color='k', lw=0.4, alpha=0.5); ax.grid(alpha=0.3)

    for k in range(2 + len(js_rest), len(axes)):
        axes[k].set_visible(False)

    # Legend
    from matplotlib.lines import Line2D
    handles = [Line2D([], [], marker='o', linestyle='', ms=6,
                       color=cmap_k(ki / max(1, k_max - k_min)),
                       label=f'k={k_min + ki}')
               for ki in range(k_max - k_min + 1)]
    fig.legend(handles=handles, loc='lower center',
               bbox_to_anchor=(0.5, -0.02), ncol=k_max - k_min + 1,
               fontsize=9, frameon=False)
    for ax in axes:
        ax.tick_params(labelsize=7)
    tag = f'  —  {iter_label}' if iter_label else ''
    fig.suptitle(f'{title_root}{tag}', fontsize=12)
    return fig

def plot_iter_stability_heatmap(W_prev, W_last, kj_list,
                                k_min, k_max, iZs_arr, title):
    """Heatmap of RMS(iter_last - iter_prev) per (k, j) cell."""
    n_k = k_max - k_min + 1
    n_j = len(iZs_arr)
    grid = np.full((n_k, n_j), np.nan)
    for m, (k, j) in enumerate(kj_list):
        diff = W_last[:, m] - W_prev[:, m]
        good = diff[np.isfinite(diff)]
        if len(good) < 3:
            continue
        ki = int(k) - k_min
        ji = int(np.where(iZs_arr == int(j))[0][0])
        grid[ki, ji] = float(np.sqrt(np.mean(good ** 2)))
    fig, ax = plt.subplots(figsize=(0.45 * n_j + 2.5, 0.6 * n_k + 1.8),
                           layout='constrained')
    vmax = float(np.nanmax(grid)) if np.any(np.isfinite(grid)) else 1.0
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1e-3
    im = ax.imshow(grid, aspect='auto', cmap='magma_r',
                   vmin=0.0, vmax=vmax)
    ax.set_xticks(range(n_j))
    ax.set_xticklabels([f'Z{j}' for j in iZs_arr], fontsize=8)
    ax.set_yticks(range(n_k))
    ax.set_yticklabels([f'k={k_min + ki}' for ki in range(n_k)])
    ax.set_xlabel('pupil Noll j')
    ax.set_ylabel('focal Noll k')
    ax.set_title(title)
    for ki in range(n_k):
        for ji in range(n_j):
            v = grid[ki, ji]
            if not np.isfinite(v):
                continue
            ax.text(ji, ki, f'{v:.3f}', ha='center', va='center',
                    fontsize=6,
                    color='white' if v > 0.55 * vmax else 'black')
    plt.colorbar(im, ax=ax, label='RMS Δ  (μm)')
    return fig


def compute_per_visit_sigmas(donut_df, residuals, fit_rows,
                             j_list, iZidx):
    """For each good visit, return per-j sigma_std and sigma_MAD arrays.

    Returns a dict with keys 'visit_ordinal', 'day_obs', 'seq_num',
    'sigma' (n_visits, n_j), 'sigma_mad' (n_visits, n_j), and the same
    pooled across j as 'sigma_pool', 'sigma_mad_pool'.
    """
    dobs_arr = np.asarray(donut_df['day_obs'])
    snum_arr = np.asarray(donut_df['seq_num'])
    good = [r for r in fit_rows if not r.get('bad_fit', False)]
    n = len(good)
    n_j = len(j_list)
    sig = np.full((n, n_j), np.nan)
    smd = np.full((n, n_j), np.nan)
    pool = np.full(n, np.nan)
    pool_m = np.full(n, np.nan)
    dobs_out = np.zeros(n, dtype=int)
    snum_out = np.zeros(n, dtype=int)
    for v, r in enumerate(good):
        d = int(r['day_obs']); s = int(r['seq_num'])
        dobs_out[v] = d
        snum_out[v] = s
        mask = (dobs_arr == d) & (snum_arr == s)
        if not np.any(mask):
            continue
        for ji, j in enumerate(j_list):
            res = residuals[mask][:, iZidx[j]]
            res = res[np.isfinite(res)]
            if len(res) < 5:
                continue
            sig[v, ji] = float(np.std(res))
            mad = float(np.median(np.abs(res - np.median(res))))
            smd[v, ji] = 1.4826 * mad
        # pooled = sqrt(SUM σ_j^2) over j with finite σ (quadrature sum)
        sj = sig[v]
        sj = sj[np.isfinite(sj)]
        if sj.size:
            pool[v] = float(np.sqrt(np.sum(sj ** 2)))
        mj = smd[v]
        mj = mj[np.isfinite(mj)]
        if mj.size:
            pool_m[v] = float(np.sqrt(np.sum(mj ** 2)))
    return dict(visit_ordinal=np.arange(n),
                day_obs=dobs_out, seq_num=snum_out,
                sigma=sig, sigma_mad=smd,
                sigma_pool=pool, sigma_mad_pool=pool_m)


def example_visit_index(sigmas):
    """Pick the visit with median pooled σ.  Falls back to first visit."""
    pool = sigmas['sigma_pool']
    good = np.where(np.isfinite(pool))[0]
    if good.size == 0:
        return 0
    return int(good[np.argsort(pool[good])[len(good) // 2]])


def plot_example_visit_histograms(donut_df, residuals, fit_rows,
                                  j_list, iZidx, example_idx,
                                  hist_range=(-1.0, 1.0), n_bins=60):
    """Single page: residual histograms for one visit, one panel per j."""
    dobs_arr = np.asarray(donut_df['day_obs'])
    snum_arr = np.asarray(donut_df['seq_num'])
    good = [r for r in fit_rows if not r.get('bad_fit', False)]
    if not good:
        return None
    r = good[min(example_idx, len(good) - 1)]
    d = int(r['day_obs']); s = int(r['seq_num'])
    mask = (dobs_arr == d) & (snum_arr == s)

    ncols = 4
    n = len(j_list)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(3.2 * ncols, 2.4 * nrows),
                             layout='constrained')
    axes = np.atleast_2d(axes).ravel()
    for ji, j in enumerate(j_list):
        ax = axes[ji]
        res = residuals[mask][:, iZidx[j]]
        res = res[np.isfinite(res)]
        if len(res) >= 5:
            ax.hist(res, bins=n_bins, range=hist_range,
                    color='steelblue', alpha=0.85)
            sigma = float(np.std(res))
            mad = float(np.median(np.abs(res - np.median(res))))
            ax.set_title(f'Z{j}  σ={sigma:.3f}  σ_MAD={1.4826 * mad:.3f}',
                         fontsize=8)
        else:
            ax.set_title(f'Z{j}  (n<5)', fontsize=8)
        ax.axvline(0, color='k', lw=0.4, alpha=0.5)
        ax.tick_params(labelsize=7)
    for k in range(n, len(axes)):
        axes[k].set_visible(False)
    fig.suptitle(f'Per-donut residual histograms  —  '
                 f'visit {d}/{s} (example_idx = {example_idx})',
                 fontsize=11)
    return fig


def plot_sigma_vs_visit_grid(sigmas, j_list, iZidx, visits_table,
                             which='sigma', title_root='σ vs visit'):
    """One page, 4×6 grid (24 panels, 21 used) of σ_j vs visit ordinal.

    Uses the shared intrinsics_lib marker scheme (elev = colour,
    rotator angle = shape, band = edge colour).  Pooled panel uses
    the quadrature-sum pool.
    """
    yvals_all = sigmas[which]
    pool      = (sigmas['sigma_pool']
                 if which == 'sigma' else sigmas['sigma_mad_pool'])
    n_v = yvals_all.shape[0]
    ordinal = sigmas['visit_ordinal']

    # Look up the standard marker style for every visit in the σ array.
    marker_lookup = build_visit_marker_lookup(visits_table)
    styles = []
    for d, s in zip(sigmas['day_obs'], sigmas['seq_num']):
        cls = marker_lookup.get((int(d), int(s)),
                                {'elev': None, 'rot': None, 'band': None})
        styles.append(visit_marker_style(elev=cls['elev'], rot=cls['rot'],
                                         band=cls['band'], base_size=5))

    ncols = 4
    nrows = 6
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 14),
                             layout='constrained', sharex=True)
    axes = axes.ravel()
    # Pooled panel
    ax = axes[0]
    for i in range(n_v):
        y = pool[i]
        if not np.isfinite(y):
            continue
        ax.plot(ordinal[i], y, **styles[i])
    ax.set_title('pooled = sqrt( Σ σ_j² )  (quadrature sum)', fontsize=9)
    ax.set_ylim(bottom=0); ax.grid(alpha=0.3)
    for ji, j in enumerate(j_list):
        ax = axes[ji + 1]
        for i in range(n_v):
            y = yvals_all[i, ji]
            if not np.isfinite(y):
                continue
            ax.plot(ordinal[i], y, **styles[i])
        ax.set_title(f'Z{j}', fontsize=8)
        ax.set_ylim(bottom=0); ax.grid(alpha=0.3); ax.tick_params(labelsize=7)
    for k in range(len(j_list) + 1, len(axes)):
        axes[k].set_visible(False)
    fig.suptitle(title_root, fontsize=12)
    return fig


# ======================================================================
# Validation marker scheme + visit lookup + residual computer.
# (Moved here from the retired validation-cell so the per-path
# validation cells can call them.)
# ======================================================================
_ELEV_PALETTE = {
    '<35':     'tab:gray',
    '35-50':   'tab:blue',
    '50-65':   'tab:green',
    '65-80':   'tab:orange',
    '>=80':    'tab:red',
    'unknown': 'lightgray',
}
_ROT_MARKERS = {
    '|rot|<=3':    'o',
    '3<|rot|<=45': 's',
    '|rot|>45':    '^',
    'unknown':     'x',
}


def _elev_group(alt_deg):
    if not np.isfinite(alt_deg):
        return 'unknown'
    if alt_deg < 35: return '<35'
    if alt_deg < 50: return '35-50'
    if alt_deg < 65: return '50-65'
    if alt_deg < 80: return '65-80'
    return '>=80'


def _rot_group(rot_deg):
    if not np.isfinite(rot_deg):
        return 'unknown'
    r = abs(rot_deg)
    if r <= 3:   return '|rot|<=3'
    if r <= 45:  return '3<|rot|<=45'
    return '|rot|>45'


def _build_visit_lookup(visits_table):
    """{(day_obs, seq_num): {alt_deg, rot_deg}} keyed by visit."""
    if visits_table is None:
        return {}
    cols = visits_table.colnames
    dobs = np.asarray(visits_table['day_obs']).tolist()
    snum = np.asarray(visits_table['seq_num']).tolist()
    if 'alt' in cols:
        alt = np.asarray(visits_table['alt'], dtype=float)
        if np.nanmax(np.abs(alt)) < 2.0 * np.pi + 1e-3:
            alt = np.rad2deg(alt)
    else:
        alt = np.full(len(visits_table), np.nan)
    rot_col = ('rotator_angle' if 'rotator_angle' in cols
               else 'rotAngle' if 'rotAngle' in cols
               else None)
    rot = (np.asarray(visits_table[rot_col], dtype=float)
           if rot_col else np.full(len(visits_table), np.nan))
    return {(int(d), int(s)): {'alt_deg': float(a), 'rot_deg': float(r)}
            for d, s, a, r in zip(dobs, snum, alt, rot)}


def compute_validation_residual(donut_df, result, coord_sys, iZs):
    """Per-donut residual after subtracting iter-final intrinsic AND
    iter-final DZ fit."""
    final_iter = result['iter_results'][-1]
    wfd_sub = final_iter['wfd_subtracted']
    measured_grid = final_iter['measured_grid']
    xcent, ycent = result['xcent'], result['ycent']
    thx = np.rad2deg(np.asarray(donut_df[f'thx_{coord_sys}'], dtype=float))
    thy = np.rad2deg(np.asarray(donut_df[f'thy_{coord_sys}'], dtype=float))
    intrinsic_at_donut = interpolate_grid_at_donuts(
        measured_grid, xcent, ycent, thx, thy, iZs)
    return wfd_sub - intrinsic_at_donut


# ======================================================================
# OFC Degree-of-Freedom (DOF) helpers for Path A diagnostics
# ======================================================================

LABELS_50DOF = [
    'M2_dz', 'M2_dx', 'M2_dy', 'M2_rx', 'M2_ry',
    'Cam_dz', 'Cam_dx', 'Cam_dy', 'Cam_rx', 'Cam_ry',
    'B1_1', 'B1_2', 'B1_3', 'B1_4', 'B1_5',
    'B1_6', 'B1_7', 'B1_8', 'B1_9', 'B1_10',
    'B1_11', 'B1_12', 'B1_13', 'B1_14', 'B1_15',
    'B1_16', 'B1_17', 'B1_18', 'B1_19', 'B1_20',
    'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5',
    'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B2_10',
    'B2_11', 'B2_12', 'B2_13', 'B2_14', 'B2_15',
    'B2_16', 'B2_17', 'B2_18', 'B2_19', 'B2_20',
]
DOF_UNITS_50 = (
    ['μm', 'μm', 'μm', 'arcsec', 'arcsec'] +           # M2 hex
    ['μm', 'μm', 'μm', 'arcsec', 'arcsec'] +           # Cam hex
    ['μm'] * 20 +                                       # M1M3 bending
    ['μm'] * 20                                         # M2 bending
)


def recover_dof_per_visit(A_modes, V, Sigma, N_diag, n_keep):
    """Recover physical DOF estimates per visit from U-mode amplitudes.

    With S = S_orig @ N (right-side normalized), SVD = U Σ V^T:
        A     = U_eff^T @ w_raw                          (μm, mode amps)
        dof_n = V_eff @ diag(1/σ_eff) @ A                (normalized)
        dof   = N @ dof_n                                (physical units)

    Parameters
    ----------
    A_modes : ndarray (n_visits, n_keep)
        U-mode amplitudes per visit (= U_effᵀ @ w_raw).
    V : ndarray (n_dof, n_dof)
        Right singular vectors of S.
    Sigma : ndarray (n_dof,)
        Singular values.
    N_diag : ndarray (n_dof,)
        OFC normalization weights (diagonal of N).
    n_keep : int
        Number of U-modes retained.

    Returns
    -------
    dof : ndarray (n_visits, n_dof)
        Physical DOF (mixed units per `DOF_UNITS_50`).
    """
    V_eff = V[:, :n_keep]                         # (n_dof, n_keep)
    inv_sig = 1.0 / Sigma[:n_keep]                # (n_keep,)
    A = np.where(np.isfinite(A_modes), A_modes, 0.0)
    # dof_norm = (V_eff * inv_sig) @ A.T  -> (n_dof, n_visits)
    dof_norm = (V_eff * inv_sig[None, :]) @ A.T
    dof_phys = N_diag[:, None] * dof_norm
    return dof_phys.T                             # (n_visits, n_dof)


def plot_dof_vs_visit_pages(dof_array, dof_labels, dof_units,
                            visits_table, sigmas_for_visit_order,
                            title_root, panels_per_page=25):
    """Two pages of 5×5 DOF panels showing DOF value vs visit ordinal.

    `dof_array` shape: (n_visits, n_dof).  Visit order matches
    `sigmas_for_visit_order['day_obs']` / `['seq_num']`.
    """
    n_v, n_dof = dof_array.shape
    ordinal = np.arange(n_v)

    marker_lookup = build_visit_marker_lookup(visits_table)
    styles = []
    for d, s in zip(sigmas_for_visit_order['day_obs'],
                     sigmas_for_visit_order['seq_num']):
        cls = marker_lookup.get((int(d), int(s)),
                                {'elev': None, 'rot': None, 'band': None})
        styles.append(visit_marker_style(elev=cls['elev'], rot=cls['rot'],
                                         band=cls['band'], base_size=5))

    figs = []
    n_pages = (n_dof + panels_per_page - 1) // panels_per_page
    for page in range(n_pages):
        lo = page * panels_per_page
        hi = min(lo + panels_per_page, n_dof)
        fig, axes = plt.subplots(5, 5, figsize=(15, 14),
                                 layout='constrained', sharex=True)
        axes = axes.ravel()
        for slot in range(25):
            dof_i = lo + slot
            ax = axes[slot]
            if dof_i >= hi:
                ax.set_visible(False); continue
            yvals = dof_array[:, dof_i]
            for i in range(n_v):
                y = yvals[i]
                if not np.isfinite(y):
                    continue
                ax.plot(ordinal[i], y, **styles[i])
            ax.set_title(f'{dof_labels[dof_i]}  ({dof_units[dof_i]})',
                         fontsize=8)
            ax.axhline(0, color='k', lw=0.4, alpha=0.5)
            ax.grid(alpha=0.3); ax.tick_params(labelsize=7)
        fig.suptitle(f'{title_root}  —  page {page + 1}/{n_pages}',
                     fontsize=12)
        figs.append(fig)
    return figs


def plot_dof_median_summary(dof_per_iter, dof_labels, dof_units,
                            title='DOF median per iteration'):
    """4-panel layout (Hex Translations / Hex Rotations / M1M3 / M2)
    showing the per-visit median DOF for each iteration as separate colours.

    `dof_per_iter` is a dict {iter_idx_0based: (n_visits, n_dof)}.
    """
    n_iter = len(dof_per_iter)
    iters = sorted(dof_per_iter.keys())
    medians = {it: np.nanmedian(dof_per_iter[it], axis=0) for it in iters}

    # DOF index buckets
    hex_trans_idx = [0, 1, 2,  5, 6, 7]               # M2 z/x/y, Cam z/x/y
    hex_rot_idx   = [3, 4,     8, 9]                  # M2 rx/ry, Cam rx/ry
    m1m3_idx      = list(range(10, 30))
    m2_idx        = list(range(30, 50))

    fig, axes = plt.subplots(4, 1, figsize=(15, 14),
                             layout='constrained',
                             gridspec_kw=dict(height_ratios=[1.0, 1.0, 1.5, 1.5]))
    iter_colors = plt.get_cmap('viridis')(np.linspace(0.1, 0.9, n_iter))

    def _panel(ax, idx_list, title, y_unit):
        x = np.arange(len(idx_list))
        # Light shaded bands for visual grouping
        for xi in range(len(idx_list)):
            if xi % 2:
                ax.axvspan(xi - 0.5, xi + 0.5, color='black', alpha=0.05)
        for ci, it in enumerate(iters):
            ax.plot(x, medians[it][idx_list], 'o',
                    ms=7, color=iter_colors[ci],
                    label=f'iter {it + 1}')
        ax.axhline(0, color='gray', lw=0.5, alpha=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels([dof_labels[i] for i in idx_list],
                           rotation=45, ha='right', fontsize=8)
        ax.set_ylabel(f'DOF Value ({y_unit})')
        ax.set_title(title)
        ax.grid(axis='y', alpha=0.3)

    _panel(axes[0], hex_trans_idx, 'Hexapod Translations', 'μm')
    _panel(axes[1], hex_rot_idx,   'Hexapod Rotations',   'arcsec')
    _panel(axes[2], m1m3_idx,      'M1M3 Bending Modes',   'μm')
    _panel(axes[3], m2_idx,        'M2 Bending Modes',     'μm')
    axes[0].legend(loc='upper right', fontsize=9)
    fig.suptitle(title, fontsize=13)
    return fig



<a id='data'></a>
## 4. Data Access — multi-chunk + visit filters

In [ ]:
# Resolve the donut parquet list (one per pipeline chunk).
donut_parquet_paths = resolve_chunk_parquets(binning, donut_parquets)
print(f'Using binning = {binning!r};  {len(donut_parquet_paths)} chunk(s):')
for p in donut_parquet_paths:
    print(f'   {p}')

donut_full, visits_full = load_chunks(donut_parquet_paths)
print(f'\nTotal: {len(donut_full)} donuts, {len(visits_full)} visits')

# Apply visit-level filters (date/elevation/rotator/seq/band).  All
# downstream sections see only the surviving visits.
visits_kept = apply_visit_filters(
    visits_full,
    day_obs_min=day_obs_min, day_obs_max=day_obs_max,
    alt_min_deg=alt_min_deg, alt_max_deg=alt_max_deg,
    rotator_min_deg=rot_min_deg, rotator_max_deg=rot_max_deg,
    seq_num_min=seq_num_min, seq_num_max=seq_num_max,
)
if allowed_bands:
    before = len(visits_kept)
    visits_kept = filter_visits_by_band(visits_kept, allowed_bands)
    print(f'Visits after band cut ({allowed_bands}): '
          f'{len(visits_kept)}/{before}')
print(f'Visits after all filters: {len(visits_kept)}/{len(visits_full)}')

if max_visits and max_visits > 0 and len(visits_kept) > max_visits:
    # Pick evenly-spaced indices so the cap samples the full filtered
    # range rather than just an end of it.
    idx = np.linspace(0, len(visits_kept) - 1, max_visits)
    idx = np.unique(np.round(idx).astype(int))
    visits_kept = visits_kept[idx]
    print(f'Visits after max_visits cap ({max_visits}): {len(visits_kept)}')

donut_df = filter_donuts_by_visits(donut_full, visits_kept)
print(f'Donuts after filters: {len(donut_df)}/{len(donut_full)}')

# Derive Noll indices from the data
nZk = np.stack(donut_df[f'zk_{coord_sys}'].values).shape[1]
noll_arr = (np.array(visits_kept['nollIndices'][0])
            if 'nollIndices' in visits_kept.colnames else None)
iZs, iZidx = derive_noll_indices(nZk, noll_arr)
print(f'Pupil Noll indices ({len(iZs)}): {iZs}')

# For the multi-chunk donut frame, use the first chunk's stem as the
# label seed (output_dir_default uses Path(...).stem only).
primary_donut_parquet = str(donut_parquet_paths[0])


<a id='z4height'></a>
## 5. Z4 Height Correction (precompute, intrinsic-only)

Compute the per-donut Z4 contribution from the focal-plane height map
**before** any DZ fitting or binning:

* `Z4hgt`           = `15 μm/mm × height(fpx, fpy)` — for the measured data.
* `Z4hgt_transpose` = `15 μm/mm × height(fpx_swap, fpy_swap)` where
  `(fpx_swap, fpy_swap)` is the per-CCD x↔y transpose around the
  detector center — for the *tabulated* intrinsic, mirroring the bug
  in the pipeline's per-CCD intrinsic Zernike calculation.

If the metrology FITS file is unavailable or the LSST stack isn't
loaded, the corrections are skipped.  When `intrinsic_transpose_bug` is
False the tabulated correction uses `Z4hgt` directly (no transpose).

In [ ]:
data_offset = None
intrinsic_offset = None
Z4hgt = None
Z4hgt_intrinsic = None

_z4_can_correct = (_ccd_height_ok and height_map_fits
                   and Path(height_map_fits).exists() and 4 in iZs)

if _z4_can_correct:
    print(f'Loading metrology: {height_map_fits}')
    metrology = make_metrology_table(height_map_fits)
    print(f'  {len(metrology)} metrology points across '
          f'{len(set(metrology["det"]))} sensors')

    from lsst.obs.lsst import LsstCam       # local import (LSST stack only)
    camera = LsstCam.getCamera()

    factor = (height_to_z4_factor
              if height_to_z4_factor is not None
              else HEIGHT_TO_Z4_UM_PER_MM)
    interp = get_height_interpolator(metrology)

    # Per-donut focal-plane (mm) coords
    fpx, fpy = compute_fp_coords(donut_df, camera)

    # Z4hgt: ideal (un-buggy) per-donut Z4 contribution from the
    # focal-plane height map.
    Z4hgt = height_to_z4(interp(fpx, fpy), factor=factor)
    print(f'Z4hgt           per donut (ideal):       '
          f'mean={float(np.nanmean(Z4hgt)):+.4f} μm  '
          f'std={float(np.nanstd(Z4hgt)):.4f} μm')

    # Z4hgt_intrinsic: the SAME height correction the pipeline used to
    # build the tabulated intrinsic — which (currently) has the per-CCD
    # x<->y transpose bug.  Evaluate the interpolator at the transposed
    # coordinates so we know what bogus Z4 was baked in.
    if intrinsic_transpose_bug:
        det_names = np.asarray(donut_df['detector']).astype(str)
        fpx_t, fpy_t = transpose_around_ccd_centers(
            fpx, fpy, det_names, camera)
        Z4hgt_intrinsic = height_to_z4(interp(fpx_t, fpy_t), factor=factor)
        print(f'Z4hgt_intrinsic per donut (transpose): '
              f'mean={float(np.nanmean(Z4hgt_intrinsic)):+.4f} μm  '
              f'std={float(np.nanstd(Z4hgt_intrinsic)):.4f} μm')
    else:
        Z4hgt_intrinsic = Z4hgt
        print('intrinsic_transpose_bug=False — Z4hgt_intrinsic = Z4hgt')

    # Correction applied to the *intrinsic* only (data is left alone):
    #   intrinsic_new = intrinsic_orig + Z4hgt - Z4hgt_intrinsic
    # i.e. peel off the buggy correction the pipeline added, then add
    # the ideal one.  build_measured_intrinsic / _uconstrained accept an
    # `intrinsic_offset` dict that is SUBTRACTED, so we pass the
    # negative of (Z4hgt - Z4hgt_intrinsic).
    intrinsic_offset = {4: Z4hgt_intrinsic - Z4hgt}
    print('Z4 height correction applied to intrinsic only:')
    print('  intrinsic_new = intrinsic_orig + Z4hgt - Z4hgt_intrinsic')
    print('  data           is unchanged')
else:
    if not _ccd_height_ok:
        print('Skipping Z4 height correction: ccd_height module unavailable')
    elif not height_map_fits:
        print('Skipping Z4 height correction: height_map_fits is None')
    elif not Path(height_map_fits).exists():
        print(f'Skipping Z4 height correction: {height_map_fits} not found')
    elif 4 not in iZs:
        print('Skipping Z4 height correction: Z4 not in iZs')


<a id='z4val'></a>
## 6. Z4 Height Correction — Validation

For one example visit, bin the Z4 intrinsic on the focal plane both
*before* and *after* the height-map correction.  The "before" panel
shows the per-CCD x↔y transposed correction baked in by the pipeline;
the "after" panel applies the fix
`intrinsic_new = intrinsic_orig + Z4hgt − Z4hgt_intrinsic`.


In [ ]:
# ============================================================
# Validation map: Z4 intrinsic (original vs Z4hgt-corrected) for one visit.
# ============================================================
if 4 in iZs and Z4hgt is not None and Z4hgt_intrinsic is not None:
    _z4_demo_dobs = int(visits_kept['day_obs'][0])
    _z4_demo_snum = int(visits_kept['seq_num'][0])
    _z4_mask = ((donut_df['day_obs'] == _z4_demo_dobs)
                & (donut_df['seq_num'] == _z4_demo_snum)).values

    _thx_v = np.rad2deg(np.asarray(donut_df.loc[_z4_mask,
                                                f'thx_{coord_sys}'],
                                   dtype=float))
    _thy_v = np.rad2deg(np.asarray(donut_df.loc[_z4_mask,
                                                f'thy_{coord_sys}'],
                                   dtype=float))
    _z4_orig = np.stack(donut_df.loc[_z4_mask,
                                     f'zk_intrinsic_{coord_sys}'].values
                        ).astype(float)[:, iZidx[4]]
    _delta   = np.asarray(Z4hgt[_z4_mask], dtype=float) - \
               np.asarray(Z4hgt_intrinsic[_z4_mask], dtype=float)
    _z4_fix  = _z4_orig + _delta

    _grid_orig, _xb, _yb, _xc, _yc = bin_median_focal(
        _thx_v, _thy_v, _z4_orig[:, None], {4: 0},
        n_bins=n_bins, fp_radius=fp_radius_grid)
    _grid_fix, *_ = bin_median_focal(
        _thx_v, _thy_v, _z4_fix[:, None], {4: 0},
        n_bins=n_bins, fp_radius=fp_radius_grid)
    _grid_dlt, *_ = bin_median_focal(
        _thx_v, _thy_v, _delta[:, None], {4: 0},
        n_bins=n_bins, fp_radius=fp_radius_grid)

    fig_z4val, axes = plt.subplots(1, 3, figsize=(15, 4.5),
                                   layout='constrained')
    vmax = float(np.nanpercentile(np.abs(_grid_orig[4]), 98))
    for ax, (g, ttl) in zip(axes, [
            (_grid_orig[4], 'Z4 intrinsic (original)'),
            (_grid_fix[4],  'Z4 intrinsic (corrected)'),
            (_grid_dlt[4],  'Δ = +Z4hgt − Z4hgt_intrinsic'),
        ]):
        v = vmax if 'Δ' not in ttl else float(np.nanpercentile(
            np.abs(g[np.isfinite(g)]), 99))
        if not np.isfinite(v) or v == 0:
            v = 1e-3
        im = ax.imshow(g.T, origin='lower', cmap='RdBu_r',
                       vmin=-v, vmax=+v,
                       extent=[_xb[0], _xb[-1], _yb[0], _yb[-1]])
        ax.set_title(ttl, fontsize=10)
        ax.set_aspect('equal')
        ax.set_xlabel(f'thy_{coord_sys} [deg]')
        ax.set_ylabel(f'thx_{coord_sys} [deg]')
        plt.colorbar(im, ax=ax, shrink=0.85, label='μm')
    fig_z4val.suptitle(f'Z4 intrinsic validation  —  visit '
                       f'{_z4_demo_dobs}/{_z4_demo_snum}',
                       fontsize=11)
    plt.show()
else:
    print('Skipping Z4 validation map (no Z4 correction was computed).')


<a id='svd'></a>
## 7. OFC Sensitivity Matrix — SVD & Reachability (reference)

Adapted from `jk_coverage_plots.ipynb`.  We slice the OFC sensitivity
matrix `S_full` of shape `(31, 29, 50)` to the pupil-j indices in
`iZs` and the focal-k slice `k_min..k_max`, then right-multiply by the
diagonal OFC normalization matrix.  The SVD of the resulting matrix
`S` gives:

* **V** — pupil-Zernike-direction basis of the column space; each
  column of V is a unit "v-mode" in DZ space.
* **U** — DoF-direction basis of the row space; each column of U is a
  unit "u-mode" in the 50-DoF actuator/figure space.
* **σ** — singular values relating the two.

`U_eff = U[:, :n_keep]` is the projector onto the first `n_keep`
u-modes.  Right-multiplying by `U_eff @ U_eff.T` projects any DZ
coefficient vector onto the slice of DZ space reachable with `n_keep`
DoF.  The per-(k, j) **reachability**

    f_{k, j} = ||U_eff^T e_{k, j}||²

is the fraction of an isolated unit DZ basis vector that lies in
col(`S`).  This is the quantity thresholded in Path B and used as a
diagnostic in Path A.


In [ ]:
# ============================================================
# Build the OFC sensitivity-matrix SVD for the chosen (k, j) slice
# ============================================================
assert _ofc_ok, ('Need lsst.ts.ofc + yaml — make sure this notebook runs '
                 'inside the LSST conda env on RSP.')

def _find_ofc_normalization_yaml(user_path=None,
                                 name='range0.5_fwhm-0.15.yaml'):
    """Locate the OFC normalization-weights yaml inside ts_config_mttcs.

    Path is $TS_CONFIG_MTTCS_DIR/MTAOS/ofc/normalization_weights/<name>.
    An explicit `user_path` wins if it exists.
    """
    if user_path:
        p = Path(user_path)
        if not p.exists():
            raise FileNotFoundError(
                f'user-supplied ofc_normalization_yaml = {p} does not exist')
        return p

    root = os.environ.get('TS_CONFIG_MTTCS_DIR')
    if not root:
        raise RuntimeError(
            'TS_CONFIG_MTTCS_DIR is not set. Run '
            '`setup -r ~/u/LSST/packages/ts_config_mttcs -j` (or the '
            'equivalent eups setup) before launching the notebook, or '
            'set ofc_normalization_yaml explicitly in the params cell.')
    target = Path(root) / 'MTAOS' / 'ofc' / 'normalization_weights' / name
    if not target.exists():
        raise FileNotFoundError(
            f'OFC normalization yaml not found at {target}. '
            'Try `git pull` in ts_config_mttcs.')
    return target

norm_yaml_path = _find_ofc_normalization_yaml(ofc_normalization_yaml)
print(f'OFC normalization yaml: {norm_yaml_path}')
with open(norm_yaml_path) as _fp:
    normalization_weights = np.array(yaml.safe_load(_fp))
normalization_matrix = np.diag(normalization_weights)
print(f'OFC normalization: {len(normalization_weights)} weights '
      f'(min={normalization_weights.min():.3g}, '
      f'max={normalization_weights.max():.3g})')

ofc_data = OFCData('lsst')
S_full = np.asarray(ofc_data.sensitivity_matrix)        # (n_field=31, n_zk=29, n_dof=50)
print(f'S_full shape (field, pupil-j, DoF) = {S_full.shape}')

# Slice to k = k_min..k_max and pupil-j = iZs.  OFC convention is
# that the axis index *equals* the Noll index — axis 0 is the unused
# Noll-0 placeholder, axis 4 is Z4, etc.  So we slice
#   - k axis with [k_min : k_max+1]   (k_min..k_max inclusive, e.g. 1..6)
#   - pupil-j axis with iZs_arr directly (advanced indexing, so it
#     naturally skips j=20 and j=21 when those are absent from iZs).
iZs_arr = np.asarray(iZs, dtype=int)
n_k = k_max - k_min + 1
n_j = len(iZs_arr)
S_slab = S_full[k_min:k_max + 1, iZs_arr, :]            # (n_k, n_j, n_dof)
n_dof = S_slab.shape[-1]
S = S_slab.reshape(-1, n_dof) @ normalization_matrix
print(f'S (kj-row, dof-col) = {S.shape}   '
      f'[k = {k_min}..{k_max} ({n_k}), pupil-j = {iZs_arr.min()}..{iZs_arr.max()} ({n_j})]')

# Row layout matches jk_coverage_plots:  row = (k - k_min) * n_j + (j_idx).
# Build kj_grid in the same order — used by build_measured_intrinsic_uconstrained.
kj_grid = [(int(k_min + ki), int(iZs_arr[ji]))
           for ki in range(n_k) for ji in range(n_j)]

# SVD
U, Sigma, Vh = np.linalg.svd(S, full_matrices=False)
V = Vh.T
print(f'SVD: U={U.shape}, Σ={Sigma.shape} (max={Sigma[0]:.3g}, min={Sigma[-1]:.3g}), V={V.shape}')

# Effective U at the chosen n_keep
n_keep_eff = min(int(n_keep), U.shape[1])
U_eff = U[:, :n_keep_eff]
print(f'U_eff shape = {U_eff.shape}  (n_keep = {n_keep_eff})')

# Per-(k, j) reachability fraction f_{k, j} = sum_m U_eff[row, m]^2
frac_1d = (U_eff ** 2).sum(axis=1)                      # shape (n_k * n_j,)
frac_2d = frac_1d.reshape(n_k, n_j)                     # (n_k, n_j)

# Convenience labels: rows = k (outer), cols = j (inner)
dz_kj_labels  = [f'k={k_min+ki},j={iZs_arr[ji]}'
                 for ki in range(n_k) for ji in range(n_j)]

# Output label used by every file written by this notebook —
# encodes the binning choice and the U-mode truncation length.
output_label = f'{binning}_nkeep_{n_keep_eff}'
print(f'output_label = {output_label!r}')


<a id='svd-plots'></a>
### 6.1 SVD diagnostic plots — V, σ, U, U²


In [ ]:
# ------------------------------------------------------------
# V matrix (DoF rows x v-mode cols), singular spectrum,
# U matrix (DZ rows x u-mode cols), and U^2 heatmap.
# Mode index is shown 1-based (i = 1..n_keep) per our convention.
# ------------------------------------------------------------
from matplotlib.colors import TwoSlopeNorm

# DOF axis grouping for the V plot.  The 50 OFC DoF are arranged as
#   0-4   M2 Hex   (5)
#   5-9   Camera Hex (5)
#   10-29 M1M3      (20)
#   30-49 M2        (20)
DOF_GROUPS = [
    ('M2 Hex',    0,  5),
    ('Cam Hex',   5, 10),
    ('M1M3',     10, 30),
    ('M2',       30, 50),
]


def _set_dof_yaxis(ax):
    for _, start, _stop in DOF_GROUPS[1:]:
        ax.axhline(start - 0.5, color='black', lw=0.5, alpha=0.5)
    # Place group labels well to the left of the tick numbers so they
    # never overlap; use axes-fraction coords on x.
    for name, start, stop in DOF_GROUPS:
        y_data = 0.5 * (start + stop - 1)
        ax.annotate(name,
                    xy=(0, y_data), xycoords=('axes fraction', 'data'),
                    xytext=(-44, 0), textcoords='offset points',
                    ha='right', va='center', rotation=90,
                    fontsize=9)
    # Leave room on the left for the rotated labels.
    ax.figure.subplots_adjust(left=0.16)


def _set_kj_yaxis(ax, n_k, n_j, k_min, iZs_arr, fontsize=6):
    n_rows = n_k * n_j
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels([f'({k_min+ki},{iZs_arr[ji]})'
                        for ki in range(n_k) for ji in range(n_j)],
                       fontsize=fontsize)
    for ki in range(1, n_k):
        ax.axhline(ki * n_j - 0.5, color='black', lw=0.4, alpha=0.4)


def _one_based_xticks(ax, n_modes, step=None):
    """Label the SVD-mode x-axis 1..n_modes (instead of 0..n-1)."""
    if step is None:
        step = max(1, n_modes // 12)
    pos = np.arange(0, n_modes, step)
    ax.set_xticks(pos)
    ax.set_xticklabels([str(int(p + 1)) for p in pos])


n_modes_total = U.shape[1]                       # = n_dof

# --- V matrix (DOF-side recipe of each singular mode) ---------------
vmax_V = float(np.max(np.abs(V)))
fig_V, axV = plt.subplots(figsize=(8.5, 6.5), layout='constrained')
im = axV.imshow(V, aspect='auto', cmap='RdBu_r',
                norm=TwoSlopeNorm(vmin=-vmax_V, vcenter=0.0, vmax=vmax_V),
                interpolation='nearest')
axV.set_title('V — DOF-side recipe of each singular mode (i = 1..n)')
axV.set_xlabel('SVD mode index i (1-based)')
axV.set_ylabel('DoF index')
_one_based_xticks(axV, n_modes_total)
_set_dof_yaxis(axV)
axV.grid(alpha=0.1)
plt.colorbar(im, ax=axV, fraction=0.04, pad=0.02, label='coefficient')

# --- Singular spectrum ----------------------------------------------
fig_sig, axS = plt.subplots(figsize=(7.5, 4.0), layout='constrained')
axS.semilogy(np.arange(1, len(Sigma) + 1), Sigma, marker='o', ms=3)
axS.axvline(n_keep_eff + 0.5, ls='--', color='k', lw=0.8,
            label=f'n_keep = {n_keep_eff}')
axS.set_xlabel('SVD mode index i (1-based)')
axS.set_ylabel(r'$\sigma_i$')
axS.set_title('OFC sensitivity-matrix singular spectrum')
axS.grid(alpha=0.3)
axS.legend()

# --- U matrix (DZ-side pattern of each singular mode) ---------------
vmax_U = float(np.max(np.abs(U)))
fig_U, axU = plt.subplots(figsize=(7.5, 14.0), layout='constrained')
im = axU.imshow(U, aspect='auto', cmap='seismic',
                norm=TwoSlopeNorm(vmin=-vmax_U, vcenter=0.0, vmax=vmax_U),
                interpolation='nearest')
axU.set_title('U — DZ-side pattern of each singular mode (i = 1..n)')
axU.set_xlabel('SVD mode index i (1-based)')
axU.axvline(n_keep_eff - 0.5, ls='--', color='k', lw=0.8)
_set_kj_yaxis(axU, n_k, n_j, k_min, iZs_arr, fontsize=6)
_one_based_xticks(axU, n_modes_total)
axU.grid(alpha=0.1)
plt.colorbar(im, ax=axU, fraction=0.04, pad=0.02, label='coefficient')

# --- U^2 heatmap ----------------------------------------------------
U2 = U_eff ** 2
fig_U2, axU2 = plt.subplots(figsize=(7.5, 14.0), layout='constrained')
im = axU2.imshow(U2, aspect='auto', cmap='magma_r',
                 vmin=0.0, vmax=float(U2.max()), interpolation='nearest')
axU2.set_title(rf'$U_{{(k,j),i}}^2$  — energy of each DZ on each kept '
               rf'U-mode (n_keep = {n_keep_eff}; i = 1..n)')
axU2.set_xlabel('U-mode index i (1-based)')
_set_kj_yaxis(axU2, n_k, n_j, k_min, iZs_arr, fontsize=6)
_one_based_xticks(axU2, n_keep_eff)
axU2.grid(alpha=0.1)
plt.colorbar(im, ax=axU2, fraction=0.04, pad=0.02, label='energy fraction')

# Sanity: explicit projector diag equals row-sums of U^2.
_P_diag = (U_eff @ U_eff.T).diagonal()
print(f'max |P_diag - row-sum(U^2)| = '
      f'{float(np.max(np.abs(_P_diag - U2.sum(axis=1)))):.2e}')

plt.show()


<a id='reachability'></a>
### 6.2 Reachability heatmap & summary table

`f_{k, j}` is the fraction of an isolated DZ basis vector e_{k, j}
that lies in col(S).  Cells with `f >= reach_threshold` are selected
for Path B's removal spec.


In [ ]:
# ------------------------------------------------------------
# Reachability heatmap (cell text = 100 * f_{k, j}) + summary table.
# ------------------------------------------------------------
fig_reach, axR = plt.subplots(
    figsize=(max(8.0, 0.45 * n_j + 2.0), 0.6 * n_k + 2.0),
    layout='constrained')
im = axR.imshow(100.0 * frac_2d, aspect='auto', cmap='magma_r',
                vmin=0.0, vmax=100.0)
axR.set_xticks(range(n_j))
axR.set_xticklabels([f'Z{j}' for j in iZs_arr], rotation=0, fontsize=8)
axR.set_yticks(range(n_k))
axR.set_yticklabels([f'k={k_min+ki}' for ki in range(n_k)])
axR.set_xlabel('pupil Noll j')
axR.set_ylabel('focal Noll k')
axR.set_title(f'Reachability  f_{{k,j}} = ‖U_effᵀ e_{{k,j}}‖²   '
              f'(n_keep = {n_keep_eff})')
for ki in range(n_k):
    for ji in range(n_j):
        val = 100.0 * frac_2d[ki, ji]
        col = 'white' if val > 55 else 'black'
        axR.text(ji, ki, f'{val:4.0f}', ha='center', va='center',
                 fontsize=6, color=col)
plt.colorbar(im, ax=axR, label='Reachability  (%)')

# --- Per-pupil-j summary table (Mean Reachability) -------------------
mean_per_j = frac_2d.mean(axis=0)
n_pass_per_j = (frac_2d >= reach_threshold).sum(axis=0)
print(f'\nReachability summary  '
      f'(n_keep = {n_keep_eff}, threshold = {reach_threshold:g}):\n')
print('   j     mean f       # k passing')
print('   ---   ----------   -----------')
for ji, j in enumerate(iZs_arr):
    print(f'   {int(j):3d}   {mean_per_j[ji]:10.4f}   '
          f'{int(n_pass_per_j[ji]):>11d} / {n_k}')

n_pass_total = int((frac_2d >= reach_threshold).sum())
print(f'\nTotal cells passing threshold: '
      f'{n_pass_total} / {n_k * n_j}')

plt.show()


<a id='fit-demo'></a>
### 6.3 Reference example — fit a full (k = 1..6) × 21 j DZ pattern

To validate the U-mode projection on real data, we draw one measured
visit (the first kept visit) and fit *all* (k, j) cells in `kj_grid`
to its `(zk_data - zk_intrinsic_tab)` per-donut wavefront.  The raw
fit and the U-mode-projected fit are then evaluated on a regular
focal-plane grid and shown in microns, alongside the residual (raw −
projected).


In [ ]:
# ------------------------------------------------------------
# Reference fit demo on the first kept visit
# ------------------------------------------------------------
from measured_intrinsic import (_apply_uconstraint, default_removal_spec)

# Pick a demo visit (first visit in visits_kept)
_demo_dobs = int(visits_kept['day_obs'][0])
_demo_snum = int(visits_kept['seq_num'][0])
_mask_demo = ((donut_df['day_obs'] == _demo_dobs)
              & (donut_df['seq_num'] == _demo_snum)).values
print(f'Demo visit: day_obs={_demo_dobs}, seq_num={_demo_snum}, '
      f'{int(_mask_demo.sum())} donuts')

# Build coefficient parameter dict — all (k, j) in kj_grid
_thx_demo = np.rad2deg(np.asarray(donut_df.loc[_mask_demo, f'thx_{coord_sys}'],
                                  dtype=float))
_thy_demo = np.rad2deg(np.asarray(donut_df.loc[_mask_demo, f'thy_{coord_sys}'],
                                  dtype=float))
_zk_data_demo = np.stack(
    donut_df.loc[_mask_demo, f'zk_{coord_sys}'].values).astype(float)
_zk_intr_demo = np.stack(
    donut_df.loc[_mask_demo, f'zk_intrinsic_{coord_sys}'].values).astype(float)

# Build a spec covering every (k = k_min..k_max, j in iZs_arr) cell.
# expand_removal_spec expects {focal_k: [pupil_j, ...]} so we key it
# by k.  by_pupil (returned) is the inverse layout used downstream.
_full_spec = {int(k_min + ki): [int(j) for j in iZs_arr] for ki in range(n_k)}
_pairs_full, _by_pupil_full, _by_focal_full = expand_removal_spec(_full_spec)

# Fit
_params_raw, _contrib_raw = _fit_one_image_subset(
    _thx_demo, _thy_demo, _zk_data_demo, _zk_intr_demo,
    iZidx, _by_pupil_full, fp_radius_basis,
    _demo_dobs, _demo_snum, 0)

# Project onto first n_keep U-modes
_params_proj, _w_raw, _w_proj = _apply_uconstraint(
    _params_raw, kj_grid, U_eff)
_contrib_proj = _dz_contrib_from_params(
    _params_proj, _by_pupil_full, _thx_demo, _thy_demo,
    iZidx, fp_radius_basis)

# Re-bin both onto the focal grid
_grid_raw, _xb, _yb, _xc, _yc = bin_median_focal(
    _thx_demo, _thy_demo, _contrib_raw, iZidx,
    n_bins=n_bins, fp_radius=fp_radius_grid)
_grid_proj, *_ = bin_median_focal(
    _thx_demo, _thy_demo, _contrib_proj, iZidx,
    n_bins=n_bins, fp_radius=fp_radius_grid)

# Plot raw, projected, residual = raw - projected per pupil j
_nj_show = min(n_j, 6)                  # show first 6 pupil-j for compactness
_js_show = list(iZs_arr[:_nj_show])
fig_demo, axes = plt.subplots(_nj_show, 3,
                              figsize=(11.0, 2.6 * _nj_show),
                              layout='constrained')
for r, j in enumerate(_js_show):
    g_raw  = _grid_raw[int(j)]
    g_proj = _grid_proj[int(j)]
    g_res  = g_raw - g_proj
    vmax = float(np.nanpercentile(np.abs(g_raw[np.isfinite(g_raw)]), 99))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1e-3
    for c, (g, ttl) in enumerate([
            (g_raw,  f'fit  (j={int(j)})'),
            (g_proj, f'U-proj  (j={int(j)})'),
            (g_res,  f'raw − proj  (j={int(j)})')]):
        ax = axes[r, c]
        im = ax.imshow(g.T, origin='lower', cmap='RdBu_r',
                       vmin=-vmax, vmax=+vmax,
                       extent=[_xb[0], _xb[-1], _yb[0], _yb[-1]])
        ax.set_title(ttl, fontsize=9)
        ax.set_aspect('equal')
        plt.colorbar(im, ax=ax, shrink=0.85, label='μm')
fig_demo.suptitle(f'Reference fit demo  —  visit {_demo_dobs}/{_demo_snum}  '
                  f'(n_keep = {n_keep_eff})', fontsize=11)
plt.show()

# Print before/after coefficient comparison for a few key (k, j)
print('\nSample raw vs. projected coefficients (μm):')
print('   k   j    raw         projected')
for k, j in kj_grid[:8] + kj_grid[-4:]:
    rkey = f'dz_z{j}_c{k}'
    raw = _params_raw.get(rkey, np.nan)
    proj = _params_proj.get(rkey, np.nan)
    print(f'   {k}   {j:>2d}   {raw:+9.4f}   {proj:+9.4f}')


<a id='analysis-pathA'></a>
## 8. Path A — U-mode constrained DZ removal

Fit *all* (k = `k_min`..`k_max`) × `len(iZs)` pupil-j DZ coefficients
on each visit, project the coefficient vector onto the first
`n_keep` U-modes (`w_proj = U_eff @ U_effᵀ @ w`), then reconstruct
the per-donut DZ contribution from `w_proj` and subtract it from the
measured Zernike maps.  Iterates `n_iter` times.


In [ ]:
# ============================================================
# Path A — U-mode-constrained build of the measured intrinsic.
# ============================================================
result_A = build_measured_intrinsic_uconstrained(
    donut_df, visits_kept, coord_sys, iZs,
    kj_grid=kj_grid,
    U_eff=U_eff,
    n_iter=n_iter,
    n_bins=n_bins,
    fp_radius_basis=fp_radius_basis,
    fp_radius_grid=fp_radius_grid,
    min_donuts=min_donuts,
    bad_fit_threshold=bad_fit_threshold,
    data_offset=data_offset,
    intrinsic_offset=intrinsic_offset,
)
print(f'\nPath A iterations completed: {len(result_A["iter_results"])}')
for it_idx, it in enumerate(result_A['iter_results']):
    n_total = len(it['fit_rows'])
    n_bad   = sum(1 for r in it['fit_rows'] if r.get('bad_fit', False))
    n_good  = n_total - n_bad
    print(f'  iter {it_idx + 1}: {n_good}/{n_total} visits passed '
          f'(min_donuts >= {min_donuts}, |coeff| <= {bad_fit_threshold} μm)')


<a id='pathA-validation'></a>
## 9. Path A — Validation

Same skeleton as the Path B validation block, with Path-A-specific
extras:

1. `w_raw` heatmap per iteration (rows = full 6 × 21 (k, j) grid).
2. `w_fit` heatmap per iteration (U-projected coefficients).
3. `w_residual = w_raw − w_fit` heatmap per iteration.
4. U-mode amplitude heatmap per iteration + line plots for **all**
   `n_keep_eff` modes (μm units).
5. Per-`(k, j)` `w_fit` vs visit — **one page per iteration** with a
   k = 1..6 colour legend.
5b. Per-`(k, j)` robust RMS (RLM `bse`) vs visit, **last iteration
   only**.
6. Iteration stability heatmap on `w_fit`.
7. U-mode amplitude stability: iter (n−1) vs iter n overlay.
8. Example-visit residual histograms (median σ_pool visit).
9. Residual `σ` vs visit using the standard intrinsics_lib marker
   scheme; pooled panel uses the **quadrature sum** `sqrt(Σ σ_j²)`.
10. Residual `σ_MAD` vs visit, same convention.
11. Z4 and Z5 iter-progression maps inline; full set saved to
    `pathA_comparison.pdf`.
12. **OFC DOF recovery** — for each iteration, compute physical DOF
    estimates per visit via
    `dof = N · V_eff · diag(1/σ) · U_effᵀ · w_raw`.  Two 5 × 5 pages
    of "DOF vs visit ordinal" per iteration, plus one 4-panel
    summary figure showing the per-visit *median* DOF for every
    iteration as separate colours (Hex Translations / Hex Rotations
    / M1M3 Bending / M2 Bending).


In [ ]:
# ======================================================================
# Path A — Validation block
# ======================================================================

kj_list_A = list(kj_grid)
print(f'Path A validation: {len(kj_list_A)} (k, j) cells (full grid).')

pathA_validation_figs = []

# ---- 1)+2)+3) w_raw / w_fit / w_residual heatmaps per iteration -----
per_iter_W_raw_A = {}
per_iter_W_fit_A = {}
per_iter_A_modes = {}
for it_idx, it in enumerate(result_A['iter_results']):
    W_raw = stack_per_visit_coeffs(it['fit_rows_raw'], kj_list_A)
    W_fit = stack_per_visit_coeffs(it['fit_rows'],     kj_list_A)
    W_res = W_raw - W_fit
    per_iter_W_raw_A[it_idx] = W_raw
    per_iter_W_fit_A[it_idx] = W_fit
    pathA_validation_figs.append(
        plot_w_heatmap(W_raw, kj_list_A, n_k, n_j,
                       f'Path A  w_raw  —  iter {it_idx + 1}', k_min=k_min))
    pathA_validation_figs.append(
        plot_w_heatmap(W_fit, kj_list_A, n_k, n_j,
                       f'Path A  w_fit (U-projected)  —  iter {it_idx + 1}',
                       k_min=k_min))
    pathA_validation_figs.append(
        plot_w_heatmap(W_res, kj_list_A, n_k, n_j,
                       f'Path A  w_residual = raw − fit  —  '
                       f'iter {it_idx + 1}', k_min=k_min))
    A = (np.where(np.isfinite(W_raw), W_raw, 0.0) @ U_eff)
    per_iter_A_modes[it_idx] = A

# ---- 4) U-mode amplitude heatmap + ALL-mode line plots --------------
def _plot_modes_heatmap(A, title):
    n_v, n_keep_ = A.shape
    finite = A[np.isfinite(A)]
    vmax = (float(np.nanpercentile(np.abs(finite), 98))
            if finite.size else 1e-3)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1e-3
    fig, ax = plt.subplots(figsize=(11.5, 5.0), layout='constrained')
    im = ax.imshow(A.T, aspect='auto', cmap='RdBu_r',
                   vmin=-vmax, vmax=+vmax,
                   extent=[-0.5, n_v - 0.5, n_keep_ - 0.5, -0.5])
    ax.set_xlabel('Visit ordinal')
    ax.set_ylabel('U-mode index (1-based)')
    ax.set_title(title)
    yt = np.arange(0, n_keep_, max(1, n_keep_ // 12))
    ax.set_yticks(yt)
    ax.set_yticklabels([str(int(p + 1)) for p in yt])
    plt.colorbar(im, ax=ax, label='Uᵀ w_raw (μm)')
    return fig


def _plot_modes_lines_all(A, n_keep_, title):
    n_v = A.shape[0]
    ncols = 5
    nrows = (n_keep_ + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 1.7 * nrows),
                             layout='constrained', sharex=True)
    axes = np.atleast_2d(axes).ravel()
    cmap = plt.get_cmap('tab20')
    for mi in range(nrows * ncols):
        ax = axes[mi]
        if mi >= n_keep_:
            ax.axis('off'); continue
        ax.plot(np.arange(n_v), A[:, mi], '.-',
                ms=2, lw=0.6, color=cmap(mi % 20))
        ax.axhline(0, color='k', lw=0.4, alpha=0.5)
        ax.set_title(f'mode {mi + 1}', fontsize=7)
        ax.grid(alpha=0.3); ax.tick_params(labelsize=6)
    fig.suptitle(title, fontsize=11)
    return fig


for it_idx, A in per_iter_A_modes.items():
    pathA_validation_figs.append(_plot_modes_heatmap(A,
        f'Path A  U-mode amplitudes  A = Uᵀ w_raw  —  iter {it_idx + 1}  (μm)'))
    pathA_validation_figs.append(_plot_modes_lines_all(A, n_keep_eff,
        f'Path A  ALL {n_keep_eff} U-mode amplitudes vs visit  —  '
        f'iter {it_idx + 1}  (μm)'))

# ---- 5) Per-(k, j) w_fit vs visit (one page per iteration) ----------
for it_idx in range(n_iter):
    pathA_validation_figs.append(
        plot_per_kj_vs_visit_page(
            per_iter_W_fit_A[it_idx], kj_list_A, iZs_arr,
            k_min, k_max,
            title_root='Path A  —  per-(k, j) w_fit (U-projected) vs visit',
            iter_label=f'iter {it_idx + 1}'))

# ---- 5b) Robust RMS (last iteration only) ---------------------------
def _stack_per_visit_err_A(fit_rows, kj_list):
    n_v = len(fit_rows); n_m = len(kj_list)
    E = np.full((n_v, n_m), np.nan)
    for v, row in enumerate(fit_rows):
        for m, (k, j) in enumerate(kj_list):
            E[v, m] = float(row.get(f'dz_z{j}_c{k}_err', np.nan))
    return E

E_last_A = _stack_per_visit_err_A(
    result_A['iter_results'][-1]['fit_rows_raw'], kj_list_A)
pathA_validation_figs.append(
    plot_per_kj_vs_visit_page(
        E_last_A, kj_list_A, iZs_arr, k_min, k_max,
        title_root=(f'Path A  —  per-(k, j) robust RMS (RLM bse) vs visit '
                    f'(iter {n_iter})'),
        iter_label=''))

# ---- 6) Iteration stability heatmap on w_fit ------------------------
if n_iter >= 2:
    pathA_validation_figs.append(plot_iter_stability_heatmap(
        per_iter_W_fit_A[n_iter - 2], per_iter_W_fit_A[n_iter - 1],
        kj_list_A, k_min, k_max, iZs_arr,
        title=(f'Path A  iteration stability  —  RMS '
               f'(iter {n_iter} − iter {n_iter - 1}) per (k, j)  '
               f'[on w_fit]')))

# ---- 7) U-mode amplitude stability (last two iters) -----------------
if n_iter >= 2:
    A_prev = per_iter_A_modes[n_iter - 2]
    A_last = per_iter_A_modes[n_iter - 1]
    n_v = A_prev.shape[0]
    ncols = 5
    nrows = (n_keep_eff + ncols - 1) // ncols
    fig_uA, axes = plt.subplots(nrows, ncols, figsize=(15, 1.7 * nrows),
                                layout='constrained', sharex=True)
    axes = np.atleast_2d(axes).ravel()
    for mi in range(nrows * ncols):
        ax = axes[mi]
        if mi >= n_keep_eff:
            ax.axis('off'); continue
        ax.plot(np.arange(n_v), A_prev[:, mi], '.', ms=2, color='steelblue',
                alpha=0.6, label=f'iter {n_iter - 1}' if mi == 0 else None)
        ax.plot(np.arange(n_v), A_last[:, mi], '.', ms=2.5, color='crimson',
                alpha=0.9, label=f'iter {n_iter}' if mi == 0 else None)
        ax.axhline(0, color='k', lw=0.4, alpha=0.5)
        ax.set_title(f'mode {mi + 1}', fontsize=7)
        ax.grid(alpha=0.3); ax.tick_params(labelsize=6)
        if mi == 0:
            ax.legend(fontsize=6, loc='best')
    fig_uA.suptitle(
        f'Path A  iter stability  —  U-mode amplitudes  '
        f'iter {n_iter - 1} vs iter {n_iter}  (μm)', fontsize=11)
    pathA_validation_figs.append(fig_uA)

# ---- 8)+9)+10) Histograms + sigma plots -----------------------------
residuals_A = compute_validation_residual(donut_df, result_A, coord_sys, iZs)
sigmas_A = compute_per_visit_sigmas(
    donut_df, residuals_A,
    result_A['iter_results'][-1]['fit_rows'],
    list(iZs), iZidx)
ex_idx_A = example_visit_index(sigmas_A)
print(f'Path A example visit (median σ_pool): '
      f'idx {ex_idx_A}, day_obs {int(sigmas_A["day_obs"][ex_idx_A])}, '
      f'seq_num {int(sigmas_A["seq_num"][ex_idx_A])}, '
      f'σ_pool = {float(sigmas_A["sigma_pool"][ex_idx_A]):.3f} μm')

fig_hist_A = plot_example_visit_histograms(
    donut_df, residuals_A,
    result_A['iter_results'][-1]['fit_rows'],
    list(iZs), iZidx, ex_idx_A,
    hist_range=residual_hist_range, n_bins=residual_n_hist_bins)
if fig_hist_A is not None:
    pathA_validation_figs.append(fig_hist_A)

pathA_validation_figs.append(plot_sigma_vs_visit_grid(
    sigmas_A, list(iZs), iZidx, visits_kept,
    which='sigma',
    title_root='Path A  —  residual σ per pupil j vs visit ordinal'))
pathA_validation_figs.append(plot_sigma_vs_visit_grid(
    sigmas_A, list(iZs), iZidx, visits_kept,
    which='sigma_mad',
    title_root='Path A  —  residual σ_MAD per pupil j vs visit ordinal'))

del residuals_A

# ---- 11) Iter-progression maps (Z4+Z5 inline, full set to PDF) -----
xbins_A, ybins_A = result_A['xbins'], result_A['ybins']
iter_grids_all_A = _build_iter_grids_list(result_A)

pathA_comparison_figs = []
for j in iZs:
    iter_grids_for_j = [(label, gdict.get(j))
                        for label, gdict in iter_grids_all_A]
    fig = plot_iter_progression_for_j(
        j,
        result_A['original_median'].get(j),
        iter_grids_for_j,
        result_A['tabulated_median'].get(j),
        xbins_A, ybins_A, coord_sys,
        path_tag=' — Path A')
    pathA_comparison_figs.append(fig)
    if int(j) in (4, 5):
        plt.show()
    else:
        plt.close(fig)

# ---- 12) DOF recovery per iteration --------------------------------
print('\nRecovering OFC DOF estimates per visit (Path A)...')
N_diag = np.asarray(normalization_weights, dtype=float)
dof_per_iter_A = {}
for it_idx, it in enumerate(result_A['iter_results']):
    A_it = per_iter_A_modes[it_idx]
    dof = recover_dof_per_visit(A_it, V, Sigma, N_diag, n_keep_eff)
    dof_per_iter_A[it_idx] = dof
    # Two 5×5 pages per iteration
    pathA_validation_figs.extend(
        plot_dof_vs_visit_pages(
            dof, LABELS_50DOF, DOF_UNITS_50, visits_kept, sigmas_A,
            title_root=f'Path A  DOF vs visit  —  iter {it_idx + 1}'))

# DOF median summary — iterations as colours
pathA_validation_figs.append(
    plot_dof_median_summary(dof_per_iter_A, LABELS_50DOF, DOF_UNITS_50,
                            title='Path A  —  DOF median per iteration'))

print(f'Built {len(pathA_validation_figs)} Path A validation figures and '
      f'{len(pathA_comparison_figs)} iter-progression pages (Z4, Z5 inline).')


<a id='analysis-pathB'></a>
## 10. Path B — Reachability-thresholded DZ removal


In [ ]:
# ============================================================
# Path B — build_measured_intrinsic with a removal_spec selected
# by reachability threshold.
# ============================================================
if removal_spec_manual is not None:
    removal_spec_B = removal_spec_manual
    print('Path B: using user-supplied removal_spec_manual')
else:
    removal_spec_B = removal_spec_from_reachability(
        frac_2d, range(k_min, k_max + 1), iZs_arr, reach_threshold)
    print(f'Path B: removal_spec built from reachability '
          f'>= {reach_threshold:g}')

_pairs_B, _by_pupil_B, _by_focal_B = expand_removal_spec(removal_spec_B)
# Legacy aliases so downstream plots-/tracking-/ccs-maps-/validation-
# cells (originally written for a single-path setup) keep working.
pairs    = _pairs_B
by_pupil = _by_pupil_B
by_focal = _by_focal_B
print(f'\nDZ removal spec (Path B): {len(_pairs_B)} (k, j) modes')
for k in sorted(_by_focal_B):
    js = _by_focal_B[k]
    js_str = ', '.join(str(j) for j in js)
    print(f'  k={k}: ({len(js)} term(s))  j = [{js_str}]')
# Show the reachability values of cells right above / below the
# current threshold so the user can sanity-check the cutoff.
_near = []
for ki, k in enumerate(range(k_min, k_max + 1)):
    for ji, j in enumerate(iZs_arr):
        f = float(frac_2d[ki, ji])
        if abs(f - reach_threshold) < 0.05:
            _near.append((f, int(k), int(j),
                          'IN' if f >= reach_threshold else 'out'))
_near.sort(reverse=True)
if _near:
    print(f'\n  cells within ±0.05 of threshold {reach_threshold:g}:')
    for f, k, j, tag in _near[:20]:
        print(f'    f={f:6.3f}  k={k}, j={j:2d}  [{tag}]')

result_B = build_measured_intrinsic(
    donut_df, visits_kept, coord_sys, iZs, removal_spec_B,
    n_iter=n_iter,
    n_bins=n_bins,
    fp_radius_basis=fp_radius_basis,
    fp_radius_grid=fp_radius_grid,
    min_donuts=min_donuts,
    bad_fit_threshold=bad_fit_threshold,
    data_offset=data_offset,
    intrinsic_offset=intrinsic_offset,
)
# Back-compat: keep `result` pointing at Path B so the downstream
# diagnostics / output cells (originally one-path) still work.
result = result_B
print(f'\nPath B iterations completed: {len(result_B["iter_results"])}')
for it_idx, it in enumerate(result_B['iter_results']):
    n_total = len(it['fit_rows'])
    n_bad = sum(1 for r in it['fit_rows'] if r.get('bad_fit', False))
    n_good = n_total - n_bad
    print(f'  iter {it_idx + 1}: {n_good}/{n_total} visits passed '
          f'(min_donuts >= {min_donuts}, |coeff| <= {bad_fit_threshold} μm)')


<a id='pathB-validation'></a>
## 11. Path B — Validation

1. `w_raw` heatmap per iteration (rows = (k, j) cells in removal spec).
2. Per-`(k, j)` DZ-fit vs visit — **one page per iteration** with a
   k = 1..6 colour legend.  `(k=1, j=4)` Focus is broken out on its
   own panel; `k = 2..6` for `j = 4` shares a panel; the other 20
   pupil-j values each show k = 1..6 overlaid.
3. Per-`(k, j)` robust RMS (RLM `bse`) vs visit, **last iteration
   only**.
4. Iteration stability heatmap: RMS(iter_n − iter_{n−1}) per (k, j).
5. Example-visit residual histograms (median σ_pool visit).
6. Residual `σ` vs visit — standard intrinsics_lib marker scheme;
   pooled panel uses the **quadrature sum** `sqrt(Σ σ_j²)`.
7. Residual `σ_MAD` vs visit, same layout.
8. Z4 and Z5 iter-progression maps inline; full set written to
   `pathB_comparison.pdf`.


In [ ]:
# ======================================================================
# Path B — Validation block
# ======================================================================

kj_list_B = removal_spec_kj_list(_by_focal_B)
print(f'Path B validation: {len(kj_list_B)} (k, j) cells in removal spec.')

pathB_validation_figs = []

# ---- 1) w_raw heatmaps per iteration --------------------------------
for it_idx, it in enumerate(result_B['iter_results']):
    W = stack_per_visit_coeffs(it['fit_rows'], kj_list_B)
    pathB_validation_figs.append(
        plot_w_heatmap(W, kj_list_B, n_k, n_j,
                       f'Path B  w_raw  —  iter {it_idx + 1}',
                       k_min=k_min))

# ---- 2) Per-(k, j) DZ-fit vs visit  (one page per iteration) --------
for it_idx, it in enumerate(result_B['iter_results']):
    W = stack_per_visit_coeffs(it['fit_rows'], kj_list_B)
    pathB_validation_figs.append(
        plot_per_kj_vs_visit_page(
            W, kj_list_B, iZs_arr, k_min, k_max,
            title_root='Path B  —  per-(k, j) fitted DZ vs visit',
            iter_label=f'iter {it_idx + 1}'))

# ---- 3) Robust RMS per-(k, j) vs visit (last iteration only) --------
def _stack_per_visit_err(fit_rows, kj_list):
    n_v = len(fit_rows); n_m = len(kj_list)
    E = np.full((n_v, n_m), np.nan)
    for v, row in enumerate(fit_rows):
        for m, (k, j) in enumerate(kj_list):
            E[v, m] = float(row.get(f'dz_z{j}_c{k}_err', np.nan))
    return E

E_last = _stack_per_visit_err(
    result_B['iter_results'][-1]['fit_rows'], kj_list_B)
pathB_validation_figs.append(
    plot_per_kj_vs_visit_page(
        E_last, kj_list_B, iZs_arr, k_min, k_max,
        title_root=(f'Path B  —  per-(k, j) robust RMS (RLM bse) vs visit '
                    f'(iter {n_iter})'),
        iter_label=''))

# ---- 4) Iteration stability heatmap ---------------------------------
if n_iter >= 2:
    W_prev = stack_per_visit_coeffs(
        result_B['iter_results'][-2]['fit_rows'], kj_list_B)
    W_last = stack_per_visit_coeffs(
        result_B['iter_results'][-1]['fit_rows'], kj_list_B)
    pathB_validation_figs.append(plot_iter_stability_heatmap(
        W_prev, W_last, kj_list_B, k_min, k_max, iZs_arr,
        title=(f'Path B  iteration stability  —  RMS '
               f'(iter {n_iter} − iter {n_iter - 1}) per (k, j)')))

# ---- 5)+6)+7) Example-visit histograms + sigma plots ----------------
residuals_B = compute_validation_residual(donut_df, result_B, coord_sys, iZs)
sigmas_B = compute_per_visit_sigmas(
    donut_df, residuals_B,
    result_B['iter_results'][-1]['fit_rows'],
    list(iZs), iZidx)
ex_idx_B = example_visit_index(sigmas_B)
print(f'Path B example visit (median σ_pool): '
      f'idx {ex_idx_B}, day_obs {int(sigmas_B["day_obs"][ex_idx_B])}, '
      f'seq_num {int(sigmas_B["seq_num"][ex_idx_B])}, '
      f'σ_pool = {float(sigmas_B["sigma_pool"][ex_idx_B]):.3f} μm')

fig_hist_B = plot_example_visit_histograms(
    donut_df, residuals_B,
    result_B['iter_results'][-1]['fit_rows'],
    list(iZs), iZidx, ex_idx_B,
    hist_range=residual_hist_range, n_bins=residual_n_hist_bins)
if fig_hist_B is not None:
    pathB_validation_figs.append(fig_hist_B)

pathB_validation_figs.append(plot_sigma_vs_visit_grid(
    sigmas_B, list(iZs), iZidx, visits_kept,
    which='sigma',
    title_root='Path B  —  residual σ per pupil j vs visit ordinal'))
pathB_validation_figs.append(plot_sigma_vs_visit_grid(
    sigmas_B, list(iZs), iZidx, visits_kept,
    which='sigma_mad',
    title_root='Path B  —  residual σ_MAD per pupil j vs visit ordinal'))

del residuals_B

# ---- 8) Iter-progression maps (Z4+Z5 inline, full set to PDF) -------
xbins, ybins = result_B['xbins'], result_B['ybins']
iter_grids_all_B = _build_iter_grids_list(result_B)
iter2_grids = result_B['iter_results'][-1]['measured_grid']
iter1_grids = result_B['iter_results'][0]['measured_grid']

pathB_comparison_figs = []
for j in iZs:
    if j not in by_pupil:
        continue
    iter_grids_for_j = [(label, gdict.get(j))
                        for label, gdict in iter_grids_all_B]
    fig = plot_iter_progression_for_j(
        j,
        result_B['original_median'].get(j),
        iter_grids_for_j,
        result_B['tabulated_median'].get(j),
        xbins, ybins, coord_sys,
        path_tag=' — Path B')
    pathB_comparison_figs.append(fig)
    if int(j) in (4, 5):
        plt.show()
    else:
        plt.close(fig)

print(f'Built {len(pathB_validation_figs)} Path B validation figures and '
      f'{len(pathB_comparison_figs)} iter-progression pages (Z4, Z5 inline).')
comparison_figs = pathB_comparison_figs


<a id='compare-AB'></a>
## 12. Path A vs Path B — Measured Intrinsic comparison


In [ ]:
# ============================================================
# Per-pupil-j comparison of the iter-(n_iter) measured intrinsic grids
# between Path A (U-mode constrained) and Path B (reachability-thresholded).
# ============================================================
_grid_A = result_A['iter_results'][-1]['measured_grid']
_grid_B = result_B['iter_results'][-1]['measured_grid']
_xb_, _yb_ = result_A['xbins'], result_A['ybins']

# Compute a common color scale per panel column.  Δ uses a tighter
# percentile because it's typically small.
def _vmax_pct(arrs, p=99.0):
    pooled = np.concatenate(
        [a.ravel()[np.isfinite(a.ravel())] for a in arrs if a is not None])
    return float(np.nanpercentile(np.abs(pooled), p)) if pooled.size else 1e-3

_js_show = [int(j) for j in iZs]
_cols = 4
_rows = (len(_js_show) + _cols - 1) // _cols

# --- Page 1: Path A iter-final maps ---------------------------------
fig_A, axA = plt.subplots(_rows, _cols,
                          figsize=(3.0 * _cols, 2.6 * _rows),
                          layout='constrained')
axA = np.asarray(axA).reshape(_rows, _cols)
vmaxA = _vmax_pct([_grid_A[int(j)] for j in _js_show], p=98)
for idx, j in enumerate(_js_show):
    r, c = divmod(idx, _cols)
    ax = axA[r, c]
    im = ax.imshow(_grid_A[int(j)].T, origin='lower', cmap='RdBu_r',
                   vmin=-vmaxA, vmax=+vmaxA,
                   extent=[_xb_[0], _xb_[-1], _yb_[0], _yb_[-1]])
    ax.set_title(f'A  Z{j}', fontsize=9)
    ax.set_aspect('equal')
for idx in range(len(_js_show), _rows * _cols):
    axA[idx // _cols, idx % _cols].axis('off')
fig_A.colorbar(im, ax=axA.ravel().tolist(), shrink=0.6, label='μm')
fig_A.suptitle(f'Path A (U-mode constrained, n_keep={n_keep_eff}) '
               f'— iter {n_iter} measured intrinsic', fontsize=11)

# --- Page 2: Path B iter-final maps ---------------------------------
fig_B, axB = plt.subplots(_rows, _cols,
                          figsize=(3.0 * _cols, 2.6 * _rows),
                          layout='constrained')
axB = np.asarray(axB).reshape(_rows, _cols)
vmaxB = _vmax_pct([_grid_B[int(j)] for j in _js_show], p=98)
for idx, j in enumerate(_js_show):
    r, c = divmod(idx, _cols)
    ax = axB[r, c]
    im = ax.imshow(_grid_B[int(j)].T, origin='lower', cmap='RdBu_r',
                   vmin=-vmaxB, vmax=+vmaxB,
                   extent=[_xb_[0], _xb_[-1], _yb_[0], _yb_[-1]])
    ax.set_title(f'B  Z{j}', fontsize=9)
    ax.set_aspect('equal')
for idx in range(len(_js_show), _rows * _cols):
    axB[idx // _cols, idx % _cols].axis('off')
fig_B.colorbar(im, ax=axB.ravel().tolist(), shrink=0.6, label='μm')
fig_B.suptitle(f'Path B (reachability >= {reach_threshold:g}) '
               f'— iter {n_iter} measured intrinsic', fontsize=11)

# --- Page 3: A − B differences --------------------------------------
fig_D, axD = plt.subplots(_rows, _cols,
                          figsize=(3.0 * _cols, 2.6 * _rows),
                          layout='constrained')
axD = np.asarray(axD).reshape(_rows, _cols)
_diffs = [_grid_A[int(j)] - _grid_B[int(j)] for j in _js_show]
vmaxD = _vmax_pct(_diffs, p=99)
for idx, j in enumerate(_js_show):
    r, c = divmod(idx, _cols)
    ax = axD[r, c]
    im = ax.imshow(_diffs[idx].T, origin='lower', cmap='RdBu_r',
                   vmin=-vmaxD, vmax=+vmaxD,
                   extent=[_xb_[0], _xb_[-1], _yb_[0], _yb_[-1]])
    rms = float(np.nanstd(_diffs[idx]))
    ax.set_title(f'A−B  Z{j}\nrms = {rms:.3f} μm', fontsize=8)
    ax.set_aspect('equal')
for idx in range(len(_js_show), _rows * _cols):
    axD[idx // _cols, idx % _cols].axis('off')
fig_D.colorbar(im, ax=axD.ravel().tolist(), shrink=0.6, label='μm')
fig_D.suptitle('Difference  Path A − Path B  (iter-final measured intrinsic)',
               fontsize=11)

# Per-j rms summary table
print('\nRMS of (A − B) per pupil j  (μm):')
print('   j     rms')
print('   ---   --------')
for j, d in zip(_js_show, _diffs):
    print(f'   {j:3d}   {float(np.nanstd(d)):8.4f}')

plt.show()

# Stash for use in the output cell
compare_figs = [fig_A, fig_B, fig_D]


<a id='ccs-maps'></a>
## 13. Iter-final (Path B) Measured Intrinsic — CCS-binned maps

Same iter-2 measured-intrinsic data as §7 (per-donut zk in the OCS
basis after DZ subtraction), but **binned in CCS field-angle
coordinates** so CCD-fixed features stand out.

For FAM triplets taken at rotator angle ≈ 0 the CCS and OCS frames
are nearly identical, so this view differs from §7 mainly to the
extent that any rotator spread is present.  Pages have a 2×2 grid
(four pupil j per page); colour scale per panel is the local
5-95 percentile.  All `iZs` (the `nollIndices` from the fits parquet)
are covered.  Streamed directly to its own PDF.

In [ ]:
# Names for panel titles (local copy — keeps these sections self-contained).
from common.zernike_names import (
    NOLL_NAMES, NOLL_FORMULAS, FOCAL_NAMES, PUPIL_NAMES,
)
def _resolve_output_dir():
    """Resolve and create output_dir if it has not been set up yet.

    Mirrors the snippet in §11 / validation-cell so any of the
    intermediate output sections can run on its own.
    """
    global output_dir
    if output_dir is None:
        output_dir = output_dir_default(
            binning, n_keep_eff, coord_sys,
            day_obs_min, day_obs_max,
            seq_num_min, seq_num_max,
            alt_min_deg, alt_max_deg,
            rot_min_deg, rot_max_deg)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def render_intrinsic_panel_pdf(grid_dict, xbins, ybins, iZs,
                                output_pdf_path, frame_label,
                                page_subtitle,
                                panels_per_page=4, ncols=2,
                                show_first=True):
    """Stream a multi-page PDF of pupil-j panels for an iter-2 grid.

    Each page has up to `panels_per_page` panels in a `ncols`-wide grid;
    each panel uses its own 5-95 percentile color scale.  Pages are
    written one at a time and the figure is closed after savefig so
    memory stays bounded.

    Parameters
    ----------
    grid_dict : dict {pupil_j: 2D array}
    xbins, ybins : 1D arrays of bin edges
    iZs : list of pupil_j
    output_pdf_path : str / Path
    frame_label : 'OCS' or 'CCS' — used in axis labels
    page_subtitle : str — used in the per-page suptitle
    """
    n_total_pages = int(np.ceil(len(iZs) / max(1, int(panels_per_page))))
    n_pages = 0
    Path(output_pdf_path).parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf_path) as pdf:
        for start in range(0, len(iZs), panels_per_page):
            page_js = iZs[start:start + panels_per_page]
            nrows = (len(page_js) + ncols - 1) // ncols
            fig, axes = plt.subplots(
                nrows, ncols,
                figsize=(7.0 * ncols, 6.0 * nrows),
                layout='constrained')
            axes = np.atleast_1d(axes).ravel()
            for idx, j in enumerate(page_js):
                ax = axes[idx]
                grid = grid_dict.get(j)
                if grid is None or not np.any(np.isfinite(grid)):
                    ax.set_visible(False)
                    continue
                vmin, vmax = np.nanpercentile(grid, [5, 95])
                im = ax.imshow(
                    grid.T, origin='lower',
                    extent=[xbins[0], xbins[-1],
                            ybins[0], ybins[-1]],
                    cmap='viridis', interpolation='none', aspect='equal',
                    vmin=vmin, vmax=vmax)
                plt.colorbar(im, ax=ax, shrink=0.8, label='μm')
                ax.set_xlabel(f'thy_{frame_label} [deg]')
                ax.set_ylabel(f'thx_{frame_label} [deg]')
                pname = PUPIL_NAMES.get(j, f'Z{j}')
                ax.set_title(
                    f'Z{j} ({pname})  —  iter-2 measured intrinsic\n'
                    f'5-95% = [{vmin:+.3f}, {vmax:+.3f}] μm',
                    fontsize=11)
            for idx in range(len(page_js), len(axes)):
                axes[idx].set_visible(False)
            page_idx = start // panels_per_page + 1
            fig.suptitle(
                f'{page_subtitle}  ({page_idx}/{n_total_pages})',
                fontsize=12)
            if show_first and n_pages == 0:
                plt.show()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            n_pages += 1
    print(f'Wrote {n_pages} pages -> {output_pdf_path}')
    return n_pages


def _build_ccs_grid(donut_df, result, iZidx, n_bins, fp_radius):
    """Bin the iter-2 wfd_subtracted on a (thx_CCS, thy_CCS) grid.

    Excludes donuts from bad-flagged visits (uses the
    `good_donut_mask` saved by build_measured_intrinsic).
    """
    if 'thx_CCS' not in donut_df.columns or 'thy_CCS' not in donut_df.columns:
        raise KeyError(
            "donut parquet has no thx_CCS / thy_CCS columns — cannot make "
            "the CCS-binned map.  Re-run mktable to add them.")
    final_iter = result['iter_results'][-1]
    wfd_sub = final_iter['wfd_subtracted']
    gd_mask = final_iter.get(
        'good_donut_mask',
        np.ones(len(donut_df), dtype=bool))
    thx = np.rad2deg(np.asarray(donut_df['thx_CCS'], dtype=float))
    thy = np.rad2deg(np.asarray(donut_df['thy_CCS'], dtype=float))
    return bin_median_focal(
        thx[gd_mask], thy[gd_mask],
        wfd_sub[gd_mask], iZidx,
        n_bins=n_bins, fp_radius=fp_radius)


# Reuse §6 grid params unless overridden in the parameters cell.
_ccs_n_bins     = ccs_map_n_bins    or n_bins
_ccs_fp_radius  = ccs_map_fp_radius or fp_radius_grid

print('Building CCS-binned iter-2 measured-intrinsic grids...')
ccs_grid, ccs_xbins, ccs_ybins, ccs_xcent, ccs_ycent = _build_ccs_grid(
    donut_df, result, result['iZidx'],
    _ccs_n_bins, _ccs_fp_radius)
print(f'  binned {result["iZs"]!r} on a {_ccs_n_bins}x{_ccs_n_bins} CCS grid')

_resolve_output_dir()
output_pdf_ccs_maps = str(
    output_dir / 'measured_intrinsic_ccs_intrinsic_maps.pdf')

render_intrinsic_panel_pdf(
    ccs_grid, ccs_xbins, ccs_ybins, iZs,
    output_pdf_ccs_maps,
    frame_label='CCS',
    page_subtitle=('Iter-2 measured intrinsic, binned in CCS  —  '
                   'CCD-fixed features should stand out'),
    panels_per_page=ccs_maps_per_page, ncols=2,
    show_first=True)

<a id='ocs-rot0-maps'></a>
## 14. Iter-final (Path B) Measured Intrinsic — OCS, rotator near 0

Same iter-2 per-donut data (`wfd_subtracted` from §6, in the OCS basis)
binned on the OCS field-angle grid, but **restricted to donuts whose
visit has `|rotator_angle| ≤ rot0_window_deg`** (default ±3°).
Removing the rotator-spread donuts gives an OCS view that does not
suffer from rotator smearing — useful as a comparison against the
all-rotator OCS iter-2 grid in §6 / §7.

Pages have a 2×2 grid (four pupil j per page) with per-panel 5–95 %
color scaling.  Streamed directly to its own PDF
(`measured_intrinsic_ocs_rot0_intrinsic_maps.pdf`).

In [ ]:
# Build a rotator-window mask: visits whose |rotator_angle| <= rot0_window_deg.
if 'rotator_angle' not in visits_kept.colnames:
    raise KeyError(
        "visits_kept has no rotator_angle column — cannot build the "
        "OCS rot~0 view.")
vk_dobs = np.asarray(visits_kept['day_obs']).astype(int)
vk_snum = np.asarray(visits_kept['seq_num']).astype(int)
vk_rot  = np.asarray(visits_kept['rotator_angle'], dtype=float)

vk_keep = np.abs(vk_rot) <= rot0_window_deg
rot0_keys = set(zip(vk_dobs[vk_keep].tolist(),
                     vk_snum[vk_keep].tolist()))
print(f'Visits with |rotator_angle| <= {rot0_window_deg}°: '
      f'{int(vk_keep.sum())}/{len(visits_kept)}')

fd_dobs = np.asarray(donut_df['day_obs']).astype(int)
fd_snum = np.asarray(donut_df['seq_num']).astype(int)
rot0_donut_mask = np.array([
    (int(d), int(s)) in rot0_keys
    for d, s in zip(fd_dobs, fd_snum)])

# Combine with the bad-fit good_donut_mask from iter-2 so we use exactly
# the same set of "good" donuts as the OCS iter-2 grid in §6, just
# further restricted to the rotator window.
final_iter = result['iter_results'][-1]
gd_mask = final_iter.get(
    'good_donut_mask', np.ones(len(donut_df), dtype=bool))
combined_mask = rot0_donut_mask & gd_mask
print(f'Donuts kept (rotator window AND not bad-flagged): '
      f'{int(combined_mask.sum())}/{len(donut_df)}')

# Bin in OCS using exactly the §6 grid sampling.
thx = np.rad2deg(np.asarray(donut_df['thx_OCS'], dtype=float))
thy = np.rad2deg(np.asarray(donut_df['thy_OCS'], dtype=float))
wfd_sub = final_iter['wfd_subtracted']

ocs_rot0_grid, ocs_xbins, ocs_ybins, ocs_xcent, ocs_ycent = bin_median_focal(
    thx[combined_mask], thy[combined_mask],
    wfd_sub[combined_mask], result['iZidx'],
    n_bins=n_bins, fp_radius=fp_radius_grid)

_resolve_output_dir()
output_pdf_ocs_rot0 = str(
    output_dir / 'measured_intrinsic_ocs_rot0_intrinsic_maps.pdf')

render_intrinsic_panel_pdf(
    ocs_rot0_grid, ocs_xbins, ocs_ybins, iZs,
    output_pdf_ocs_rot0,
    frame_label='OCS',
    page_subtitle=(f'Iter-2 measured intrinsic, OCS  —  '
                   f'rotator within ±{rot0_window_deg:g}°'),
    panels_per_page=ccs_maps_per_page, ncols=2,
    show_first=False)

<a id='output'></a>
## 15. Output Tables

In [ ]:
if output_dir is None:
    output_dir = output_dir_default(
        binning, n_keep_eff, coord_sys,
        day_obs_min, day_obs_max,
        seq_num_min, seq_num_max,
        alt_min_deg, alt_max_deg,
        rot_min_deg, rot_max_deg)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {output_dir}')
print(f'Output label:     {output_label}')


def _save_path_outputs(result_obj, tag, intrinsic_grid_for_table):
    final_iter = result_obj['iter_results'][-1]
    dz_fits_path = output_dir / f'measured_intrinsic_{output_label}_{tag}_dz_fits.parquet'
    save_dz_fits(final_iter['fit_rows'], dz_fits_path)

    xc, yc = result_obj['xcent'], result_obj['ycent']
    tbl = assemble_intrinsic_table(
        grid=intrinsic_grid_for_table, iZs=iZs, xcent=xc, ycent=yc,
        coord_sys_grid=coord_sys, alt_coord_xform=None)
    print(f'  [{tag}] grid table: {len(tbl)} non-empty bins in {coord_sys}')

    out_path = output_dir / f'measured_intrinsic_{output_label}_{tag}_grid.{output_format}'
    if output_format == 'parquet':
        tbl.write(str(out_path), format='parquet', overwrite=True)
    elif output_format == 'fits':
        tbl.write(str(out_path), format='fits', overwrite=True)
    elif output_format == 'hdf5':
        tbl.write(str(out_path), format='hdf5', path='grid',
                  overwrite=True, append=False)
    else:
        raise ValueError(f'Unknown output_format: {output_format}')
    print(f'  [{tag}] saved measured intrinsic grid: {out_path}')


_save_path_outputs(result_B, 'pathB', iter2_grids)
_save_path_outputs(result_A, 'pathA',
                   result_A['iter_results'][-1]['measured_grid'])


def _write_pdf(figs, fname):
    p = output_dir / fname
    with PdfPages(p) as pdf:
        for fig in figs:
            pdf.savefig(fig, bbox_inches='tight')
    print(f'  wrote {p}')


print('\nWriting PDFs:')
if 'pathB_comparison_figs' in globals() and pathB_comparison_figs:
    _write_pdf(pathB_comparison_figs,
               f'measured_intrinsic_{output_label}_pathB_comparison.pdf')

if 'pathA_comparison_figs' in globals() and pathA_comparison_figs:
    _write_pdf(pathA_comparison_figs,
               f'measured_intrinsic_{output_label}_pathA_comparison.pdf')

if 'pathB_validation_figs' in globals() and pathB_validation_figs:
    _write_pdf(pathB_validation_figs,
               f'measured_intrinsic_{output_label}_pathB_validation.pdf')

if 'pathA_validation_figs' in globals() and pathA_validation_figs:
    _write_pdf(pathA_validation_figs,
               f'measured_intrinsic_{output_label}_pathA_validation.pdf')

if 'compare_figs' in globals() and compare_figs:
    _write_pdf(compare_figs,
               f'measured_intrinsic_{output_label}_pathA_vs_pathB.pdf')


<a id='format'></a>
## 16. Output Format Options — Rubin DM datasets

The "measured intrinsic" map is a **per-pupil-j focal-plane grid of
median Zernike values** — conceptually parallel to the existing batoid
intrinsic table (`zk_intrinsic_*` columns produced upstream by
`aggregateAOSVisitTable`), but data-derived.  Realistic choices:

| Option | Pros | Cons |
|---|---|---|
| **Native parquet** (default) | Matches every other AOS pipeline output; loadable by `astropy.QTable.read` and `pandas.read_parquet`; round-trips list columns natively. | Not registered with the Butler — passed by path. |
| **FITS BinTable** | DM-friendly, viewable in DS9/topcat. | Slightly heavier; list columns become variable-length arrays. |
| **HDF5 QTable** | Multi-table file possible. | Less standard for AOS outputs going forward. |
| **Custom Butler dataset type** | Full provenance, `butler.get(...)`. | Requires an RFC + StorageClass + dimension records. |

Iterating with parquet for now; once the iteration logic is stable a
Butler dataset type is a natural next step.